This script uses LDA to calculate decoding accuracy of neuron populations recorded from different sites. Decoding accuracy is calculated while controlling for number of trials and population size used from each recording. It performs the following steps:

1. Loads relevant data
    - Preprocessed spike times
    - ECoG decoding accuracy
2. Calcualtes decoding accuracy for each recording set under different parameters:
    - Event alignment (peripheral target onset, go cue, movement onset)
    - Number of trials
    - Unit adding
3. Calculates statistics between recordings as a function of ECoG decoding accuracy
4. Plot results and save figures


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import aopy
import os
import pandas as pds
from db import dbfunctions as db
from ipywidgets import interactive, widgets
import scipy
import h5py
from tqdm.auto import tqdm 
import seaborn as sn
import sklearn
from sklearn.decomposition import PCA, FactorAnalysis
from itertools import compress
import multiprocessing as mp
import time
import math
from scipy.fft import fft
import glob
from datetime import date

# Set parameters

In [3]:
save_figs = False
base_save_dir = "/media/moor-data/results/Ryan/neuropixel_targeting/"
np_preproc_data_folder = 'np_analysis_preproc_data'
ecog_dec_acc_file_name = 'ecog_decoding_maps/npinsert_ecog_decoding_all'

subject = 'affi'
align_events = ['TARGET ONSET', 'GO CUE', 'MOVEMENT ONSET']
trelevant1_time = -.1
trelevant2_time = 0.3

In [4]:
# Decoding calculation parameters
tbefore = 0.5
tafter = 1
nlda_lags = 1
niter_match = 50
ntrial_bin_size = 96
nfolds = 5

# Visualization parameters
colors = sn.color_palette(n_colors=9)
if subject == 'beignet':
    recording_brain_areas={'M1': [30, 56, 47, 40, 121, 48, 120, 98], 'PM':[11, 9, 18, 22, 10, 45]}
elif subject == 'affi':
    recording_brain_areas={'M1': [98, 74, 72, 107], 'PM':[19,29,55,58,33,10]}

plt.rcParams['xtick.labelsize']=24
plt.rcParams['ytick.labelsize']=24
plt.rcParams['axes.labelsize']=28
plt.rcParams['axes.titlesize'] = 28
plt.rcParams['axes.spines.top']=False
plt.rcParams['axes.spines.right']=False
plt.rcParams['lines.linewidth']=5


In [5]:
def xval_lda(data, labels, sample_rate, lags=3, nfolds=5, smooth_timeseries=False, smooth_width=150, return_weights=False, return_confusion_matrix=False):
    '''
        
    Args:
        data (ntime, nch, ntrials)
        labels (ntrials)
        sample_rate
        lags (int)
        nfolds (int)
        smooth_timeseries
        smooth_width (int): [ms]
        
    Returns:
        decoding_accuracy()
    '''
    ntime, nch, ntrials = data.shape 
    nlabels = len(np.unique(labels))
    
    if smooth_timeseries:
        smoothed_data = smooth_timeseries_gaus(data, sample_rate, smooth_width)
    else:
        smoothed_data = data

    # Perform n-fold xval LDA at each time point 
    decoding_accuracy = np.zeros((ntime-lags, nfolds))*np.nan
    weights = np.zeros((ntime-lags, nlabels, (1+lags)*nch, nfolds))*np.nan
    confusion_matrix = np.zeros((ntime-lags, nlabels, nlabels, nfolds))*np.nan
    nwind = smoothed_data.shape[0] - lags
    for iwind in range(nwind):
        lda = sklearn.discriminant_analysis.LinearDiscriminantAnalysis(solver='eigen', shrinkage='auto')
        # lda = sklearn.discriminant_analysis.LinearDiscriminantAnalysis()
        end_wind = iwind+lags+1
        
        # If there are nans at this timestep, return nan in all places
        if np.sum(np.isnan(smoothed_data[iwind:end_wind,:,:])):
            if nfolds < 2:
                decoding_accuracy[iwind] = np.nan
                weights[iwind,:,:] = np.nan
                confusion_matrix[iwind,:,:] = np.nan
            else:
                decoding_accuracy[iwind,:] = np.nan
                weights[iwind,:,:,:] = np.nan
                confusion_matrix[iwind,:,:,:] = np.nan

        else:
            # If not cross validated
            if nfolds < 2:
                lda.fit(np.vstack(smoothed_data[iwind:end_wind,:,:]).T - np.mean(np.vstack(smoothed_data[iwind:end_wind,:,:]).T, axis=0), labels)
                decoding_accuracy[iwind] = lda.score(np.vstack(smoothed_data[iwind:end_wind,:,:]).T - np.mean(np.vstack(smoothed_data[iwind:end_wind,:,:]).T, axis=0), labels)
                predicted_labels = lda.predict(np.vstack(smoothed_data[iwind:end_wind,:,:]).T - np.mean(np.vstack(smoothed_data[iwind:end_wind,:,:]).T, axis=0))
                confusion_matrix[iwind,:,:,0] = sklearn.metrics.confusion_matrix(labels, predicted_labels)
                                    
                                    
            # if cross validated: 
            else:
                kf = sklearn.model_selection.KFold(n_splits=nfolds,shuffle=True,random_state=None)
                for ifold, (train_idx, test_idx) in enumerate(kf.split(np.vstack(smoothed_data[iwind:end_wind,:,:]).T)):
                    lda.fit(np.vstack(smoothed_data[iwind:end_wind,:,train_idx]).T - np.mean(np.vstack(smoothed_data[iwind:end_wind,:,train_idx]).T, axis=0), labels[train_idx])
                    decoding_accuracy[iwind, ifold] = lda.score(np.vstack(smoothed_data[iwind:end_wind,:,test_idx]).T - np.mean(np.vstack(smoothed_data[iwind:end_wind,:,test_idx]).T, axis=0), labels[test_idx])
                    weights[iwind,:,:,ifold] = lda.coef_
                    predicted_labels = lda.predict(np.vstack(smoothed_data[iwind:end_wind,:,test_idx]).T - np.mean(np.vstack(smoothed_data[iwind:end_wind,:,test_idx]).T, axis=0))
                    confusion_matrix[iwind,:,:,ifold] = sklearn.metrics.confusion_matrix(labels[test_idx], predicted_labels, normalize='true')
            
    if return_weights & return_confusion_matrix:
        return decoding_accuracy, weights, confusion_matrix
    elif return_weights: 
        return decoding_accuracy, weights
    elif return_confusion_matrix:
        return decoding_accuracy, confusion_matrix
    else:
        return decoding_accuracy

def lda_single(data, labels, nfolds):
    '''
    data (ntime, nch, ntrials)
    '''
    kf = sklearn.model_selection.KFold(n_splits=nfolds,shuffle=True,random_state=1)
    lda = sklearn.discriminant_analysis.LinearDiscriminantAnalysis()
    data_out = np.zeros(nfolds)*np.nan 
    for ifold, (train_idx, test_idx) in enumerate(kf.split(np.vstack(data[:,:,:]).T)):            
        lda.fit(np.vstack(data[:,:,train_idx]).T - np.mean(np.vstack(data[:,:,train_idx]).T, axis=0), labels[train_idx])
        data_out[ifold] = lda.score(np.vstack(data[:,:,test_idx]).T - np.mean(np.vstack(data[:,:,test_idx]).T, axis=0), labels[test_idx])
    return data_out

def xval_lda_parallel(data, labels, sample_rate, lags=3, nfolds=5, smooth_timeseries=False, smooth_width=50):
    '''
    ** Needs to be able to handle just one lag **
    
    Args:
        data (ntime, nch, ntrials)
        labels (ntrials)
        sample_rate
        lags (int)
        nfolds (int)
        smooth_timeseries
        smooth_width (int): [ms]
        
    Returns:
        decoding_accuracy()
    '''
    ntime, nch, ntrials = data.shape 
    nlabels = len(np.unique(labels))
    
    if smooth_timeseries:
        smoothed_data = smooth_timeseries_gaus(data, sample_rate, smooth_width)
    else:
        smoothed_data = data
    
    # Perform n-fold xval LDA at each time point 
    decoding_accuracy = np.zeros((ntime-lags, nfolds))*np.nan
    nwind = smoothed_data.shape[0] - lags
    input_data = [smoothed_data[iwind:(iwind+lags),:,:] for iwind in range(nwind)]
    # print(len(input_data), len([labels for ii in range(nwind)]), itertools.repeat(nfolds))
    input_args = zip(input_data, [labels for ii in range(nwind)], itertools.repeat(nfolds))
    decoding_accuracy = lda_single(data, labels, nfolds)
    with Pool() as pool:
        decoding_accuracy = np.array(pool.starmap(lda_single, input_args))
    return decoding_accuracy

def smooth_timeseries_gaus(timeseries_data, samplerate, width, nstd=3, conv_mode='same'):
    '''
    Smooths across 2 
    
    Args:
        timeseries_data (ntime, ...)
        samplerate (int): Sample rate of timeseries
        width (float): Width of the gaussian in time [ms] from -nstd to +nstd
        nstd (float/int): Number of standard deviations to be used in the filter calculation.
        conv_mode (str): Sets the size of the output. Takes eithe 'full', 'valid', or 'same'. See scipy.signal.convolve for full documentationat
        
    Returns: 
        smoothed_timeseries
    '''
    sample_std = (width/nstd)*(samplerate/(1000)) # Convert from s to ms
    x = np.arange(-sample_std*nstd, nstd*sample_std+1)
    gaus_filter = (1/(sample_std*np.sqrt(2*np.pi)))*np.exp(-(x**2)/(2*sample_std**2))
    return np.apply_along_axis(scipy.signal.convolve, 0, timeseries_data, gaus_filter, mode=conv_mode, method='direct')

def get_pseudopopulation_raster(unit_df, raster_list, stable_unit_idx_list, target_idx_list, nunits):
    '''
    Args:
        unit_df [dataframe]: must have info on rec number, unitidx
        raster_list [list of (ntime, ntrials, nch) arrays]:
        stable_unit_idx_list [list of arrays]
        target_idx_list [list of (ntrial) arrays]
        nunits [int]

    Returns:
        (ntime, ntrial, nunits) pseudopopulation raster array
        (ntrial) target labels
    '''
    # Randomly select nunits from unit_df
    units_to_use = np.sort(np.random.choice(np.arange(len(unit_df)), size=nunits, replace=False))

    # Get the raster info from each of these units and compile
    ntime, ntrials, _ = raster_list[0].shape
    raster_output = np.zeros((ntime, ntrials, nunits))*np.nan
    # print(units_to_use)
    for iunit, unit in enumerate(units_to_use):
        rec_number = unit_df['rec_number'][unit]
        unit_idx = unit_df['unit_idx'][unit]
        # print(iunit, unit, rec_number, unit_idx)
        sorted_target_label_idx = np.argsort(target_idx_list[rec_number]) # Sort target labels for each unit
        raster_output[:,:,iunit] = raster_list[rec_number][:,sorted_target_label_idx,unit_idx]
        
    return raster_output, target_idx_list[rec_number][sorted_target_label_idx]
# test = get_pseudopopulation_raster(random_units[0].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=13)

# Load relevant data

## Load ECoG decoding accuracy

In [6]:
def load_hdf_group(data_dir, hdf_filename, group="/"):
    '''
    Loads any datasets from the given hdf group into a dictionary. Also will
    recursively load other groups if any exist under the given group

    Args:
        data_dir (str): folder where data is located
        hdf_filename (str): name of hdf file
        group (str): name of the group to load
    
    Returns:
        dict: all the datasets contained in the given group
    '''
    full_file_name = os.path.join(data_dir, hdf_filename)
    hdf = h5py.File(full_file_name, 'r')
    if group not in hdf:
        raise ValueError('No such group in file {}'.format(hdf_filename))

    # Recursively load groups until datasets are reached
    def _load_hdf_group(hdf):
        keys = hdf.keys()
        data = dict()
        for k in keys:
            if isinstance(hdf[k], h5py.Group):
                data[k] = _load_hdf_group(hdf[k])
            else:
                k_, v = _load_hdf_dataset(hdf[k], k)
                data[k_] = v
        return data

    data = _load_hdf_group(hdf[group])
    hdf.close()
    return data

def _load_hdf_dataset(dataset, name):
    '''
    Internal function for loading hdf datasets. Decodes json and unicode data automatically.

    Args:
        dataset (hdf object): dataset to load
        name (str): name of the dataset

    Returns:
        tuple: Tuple containing:
            | **name (str):** name of the dataset (might be modified)
            | **data (object):** loaded data
    '''
    data = dataset[()]
    if '_json' in name:
        import json
        name = name.replace('_json', '')
        data = json.loads(data)
    try:
        data = data.decode('utf-8')
    except:
        pass
    return name, data

In [7]:
# print(os.path.join(base_save_dir, ecog_dec_acc_file_name_all))
ecog_dec_acc_file_name_all = 'ecog_decoding_maps/npinsert_ecog_decoding'
ecog_dec_acc_file_name_x = 'ecog_decoding_maps/npinsert_ecog_decoding_x'
ecog_dec_acc_file_name_y = 'ecog_decoding_maps/npinsert_ecog_decoding_y'
ecog_dec_acc = load_hdf_group(base_save_dir, ecog_dec_acc_file_name_all)
ecog_dec_acc_x = load_hdf_group(base_save_dir, ecog_dec_acc_file_name_x)
ecog_dec_acc_y = load_hdf_group(base_save_dir, ecog_dec_acc_file_name_y)

In [8]:
day_colors = ecog_dec_acc[subject]['day_colors']

In [9]:
# print(ecog_dec_acc[subject].keys())
# ecog_dec_acc[subject]['rec_locations'] = np.array([[-1.99999496,  3.99998992],
#        [ 1.99999496,  3.99998992],
#        [ 1.99999496, -1.99999496],
#        [-3.99998992,  1.99999496],
#        [ 0.99999748, -0.99999748],
#        [ 2.99999244,  2.99999244],
#        [-1.99999496,  1.99999496],
#        [-1.99999496,  3.99998992],
#        [-3.99998992,  1.99999496],
#        [ 5.31998858, -0.75999837],
#        [-1.51999674,  4.55999021],
#        [-4.55999021,  3.03999347],
#        [ 4.55999021, -1.51999674],
#        [ 3.79999184,  0.75999837],
#        [ 0.        ,  3.03999347]])

## Load preprocessed neuropixel data

In [10]:
start = time.time()
aopy.utils.release_memory_limit()
df, rasters, preproc_metadata = aopy.data.base.pkl_read(f"{subject}_np_preprocessed", os.path.join(base_save_dir, np_preproc_data_folder))
print(f"{np.round((time.time()-start)/60)} min to load preprocessed data")
nrecs = preproc_metadata['nrecs']
recording_site = preproc_metadata['recording_sites'] # will be the same for all align events
implants = ['NPinsert72' if preproc_metadata['implant'][irec] == 'NP_Insert72' else 'NPinsert137' for irec in range(len(preproc_metadata['implant']))] #Rename because name in bmi3d is slightly different (TODO)
dates = np.unique(df['date'])
ntargets = len(np.unique(df['target_idx']))

KeyboardInterrupt: 

In [ ]:
# rasters['lfp'] = rasters_lfp

In [ ]:
random_units, column_units, depth_units, depth_group_info_by_site, pseudopopulation_metadata, unit_df = aopy.data.base.pkl_read(f"{subject}_np_psuedopopulations",  os.path.join(base_save_dir, np_preproc_data_folder))

In [ ]:
trelevant1 = np.where(preproc_metadata['trial_time_axis']>trelevant1_time)[0][0]
trelevant2 = np.where(preproc_metadata['trial_time_axis']>trelevant2_time)[0][0]

## Identify stable units

In [ ]:
qc_results = aopy.data.base.pkl_read(f"{subject}_QCunits", os.path.join(base_save_dir, np_preproc_data_folder))
stable_unit_labels = [qc_results['final_good_unit_labels'][irec] for irec in range(nrecs)]
stable_unit_idx = [qc_results['final_good_unit_idx'][irec] for irec in range(nrecs)]
nstable_unit = np.array([len(qc_results['final_good_unit_idx'][irec]) for irec in range(nrecs)])
neuron_pos = [qc_results['position'][irec] for irec in range(nrecs)]

# stable_unit_labels = [qc_results['manual_good_unit_labels'][irec] for irec in range(nrecs)]
# stable_unit_idx = [qc_results['manual_good_unit_idx'][irec] for irec in range(nrecs)]
# nstable_unit = np.array([len(qc_results['manual_good_unit_idx'][irec]) for irec in range(nrecs)])
# neuron_pos = [qc_results['manual_position'][irec] for irec in range(nrecs)]

## Zscore FR

In [ ]:
# Zscore the activity of each neuron (even unstable) across all trials
fr_zscore = {}
for align_event in tqdm(align_events):
    temp_spike_data = [rasters['neural'][align_event][irec][:,np.array(df['good_trial'][df['date']==dates[irec]]),:] for irec in range(nrecs)] # Shape (ntime, ntrial, nunit)
    temp_sbp_data = [rasters['sbp'][align_event][irec][:,np.array(df['good_trial'][df['date']==dates[irec]]),:] for irec in range(nrecs)] # Shape (ntime, ntrial, nunit)
    fr_zscore[align_event] = {'align_spikes_zscore': [], 'align_sbp_zscore': []}
    [fr_zscore[align_event]['align_spikes_zscore'].append((temp_spike_data[irec]-np.mean(temp_spike_data[irec], axis=(0,1)))/np.std(temp_spike_data[irec], axis=(0,1))) for irec in range(nrecs)]
    [fr_zscore[align_event]['align_sbp_zscore'].append((temp_sbp_data[irec]-np.mean(temp_sbp_data[irec], axis=(0,1)))/np.std(temp_sbp_data[irec], axis=(0,1))) for irec in range(nrecs)]

# Zscore lfp power for each frequency band
# nbands = len(list(rasters['lfp'].keys()))
# for align_event in tqdm(align_events):
#     fr_zscore[align_event]['align_lfp_zscore'] = {}
#     for iband in range(len(list(rasters['lfp'].keys()))):
#         fr_zscore[align_event]['align_lfp_zscore'][iband] = []

#         temp_lfp_data = [rasters['lfp'][iband][align_event][irec][:,np.array(df['good_trial'][df['date']==dates[irec]]),:] for irec in range(nrecs)] # Shape (ntime, ntrial, nunit)
#         [fr_zscore[align_event]['align_lfp_zscore'][iband].append((temp_lfp_data[irec]-np.nanmean(temp_lfp_data[irec], axis=(0,1)))/np.nanstd(temp_lfp_data[irec], axis=(0,1))) for irec in range(nrecs)]
        

# Calculate LDA decoding accuracy

In [ ]:
ntime = len(preproc_metadata['trial_time_axis'])
lda_results = {}
for align_event in align_events:
    lda_results[align_event] = {}
    lda_results[align_event]['sbp'] = {}
    # lda_results[align_event]['lfp'] = {}
    # for iband in range(nbands):
    #     lda_results[align_event]['lfp'][iband] = {}

## nLag analysis

In [ ]:
test_lags = np.arange(1,10)
align_event = align_events[-1]
lda_results['lag_test'] = {'scores': {}, 'weights': {}}

for irec in tqdm(range(nrecs)):
    lda_results['lag_test']['scores'][f"site{irec}"] = []
    lda_results['lag_test']['weights'][f"site{irec}"] = []
    for test_lag in tqdm(test_lags):
        units_to_use = stable_unit_idx[irec]
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        score, weights = xval_lda(np.swapaxes(fr_zscore[align_event]['align_spikes_zscore'][irec][:,:,units_to_use], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=test_lag, smooth_timeseries=True, return_weights=True)
        max_tidx = np.argmax(np.mean(score, axis=1))
        unit_rank = np.flip(np.argsort((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))))
        lda_results['lag_test']['scores'][f"site{irec}"].append(score)
        lda_results['lag_test']['weights'][f"site{irec}"].append(weights)
        # lda_results['dec_unit_weight_rank'].append((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))[unit_rank])

## All good units in all recordings

In [ ]:
# Organize data input to xval_lda
for align_event in tqdm(align_events):
    all_neural_data_list = []
    for irec in range(nrecs):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        sorted_target_label_mask = np.argsort(temp_target_labels)
        all_neural_data_list.append(fr_zscore[align_event]['align_spikes_zscore'][irec][:,sorted_target_label_mask,:][:,:,stable_unit_idx[irec]])

    lda_results[align_event]['all_units_score'], lda_results[align_event]['all_units_weight'] = xval_lda(np.swapaxes(np.concatenate(all_neural_data_list, axis=2), 1,2), temp_target_labels[sorted_target_label_mask], 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)

In [ ]:
# Unit adding spiking
groups_of_nunits_all_recs_time = np.array([2,6, 12,26,50,100, 200, len(unit_df)])
nrandom_iterations = 50
# nrandom_iterations = pseudopopulation_metadata['nrandom_groups']
target_idx_list = [np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])]) for irec in range(nrecs)] 
ntime = len(preproc_metadata['trial_time_axis'])
# for align_event in align_events:
align_event = align_events[-1]
lda_results[align_event]['unit_adding'] = {}
lda_results[align_event]['unit_adding']['scores'] = []
for igroup in tqdm(range(len(groups_of_nunits_all_recs_time))):
    # if igroup == 2:
    temp_scores = np.zeros((ntime, nfolds, nrandom_iterations))*np.nan
    # temp_weights = np.zeros((ntime, ntargets, nfolds, nrandom_iterations))*np.nan
    for iiter in tqdm(range(nrandom_iterations)):
        pseudopop_raster, target_labels = get_pseudopopulation_raster(unit_df, fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=groups_of_nunits_all_recs_time[igroup])
        a, b = xval_lda(np.swapaxes(pseudopop_raster, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        temp_scores[:,:,iiter] = a

    lda_results[align_event]['unit_adding']['scores'].append(np.mean(temp_scores,axis=1))
        # lda_results[align_event]['unit_adding']['scores'][igroup] = np.mean(temp_scores,axis=1)

In [ ]:
# Unit adding spiking
temp = np.append(np.arange(100,len(unit_df),10), len(unit_df))
# # temp = np.array([200, len(unit_df)])
groups_of_nunits_all_recs = np.concatenate((np.arange(2, 100, 2), temp))
# groups_of_nunits_all_recs = [10,200, 256]
nrandom_iterations = 100
# nrandom_iterations = pseudopopulation_metadata['nrandom_groups']
target_idx_list = [np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])]) for irec in range(nrecs)] 
ntime = len(preproc_metadata['trial_time_axis'])
align_event = align_events[-1]
# for align_event in align_events:
lda_results[align_event]['unit_adding']['scores_event'] = []
for igroup in tqdm(range(len(groups_of_nunits_all_recs))):
    temp_scores = np.zeros((nfolds, nrandom_iterations))*np.nan
    # temp_weights = np.zeros((ntime, ntargets, nfolds, nrandom_iterations))*np.nan
    for iiter in range(nrandom_iterations):
        pseudopop_raster, target_labels = get_pseudopopulation_raster(unit_df, fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=groups_of_nunits_all_recs[igroup])
        input_data = np.mean(pseudopop_raster[trelevant1:trelevant2,:,:], axis=0).reshape(1,pseudopop_raster.shape[1], pseudopop_raster.shape[2])
        a, b = xval_lda(np.swapaxes(input_data, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        temp_scores[:,iiter] = a

    lda_results[align_event]['unit_adding']['scores_event'].append(np.mean(temp_scores,axis=0))

In [ ]:
 # Single channel decoding for spikes
# for align_event in align_events:
align_event = align_events[-1]
lda_results[align_event]['single_ch_decoding'] = []
for irec in tqdm(range(nrecs)):
    nunits = len(stable_unit_idx[irec])
    temp_single_ch_decoding = np.zeros((ntime,nunits,nfolds))*np.nan
    for ich, unitid in enumerate(stable_unit_idx[irec]):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        score, _ = xval_lda(np.swapaxes(fr_zscore[align_event]['align_spikes_zscore'][irec][:,:,unitid][:,:,None], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        temp_single_ch_decoding[:,ich,:] = score
    lda_results[align_event]['single_ch_decoding'].append(temp_single_ch_decoding)

## All good units in each recording

In [ ]:
ntest_trials = fr_zscore[align_event]['align_spikes_zscore'][0].shape[1]
for align_event in align_events:
    lda_results[align_event]['dec_unit_weight_rank'] = []
    lda_results[align_event]['scores'] = []
    lda_results[align_event]['weights'] = []
    for irec in tqdm(range(nrecs)):
        try:
            units_to_use = stable_unit_idx[irec]
            temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
            score, weights = xval_lda(np.swapaxes(fr_zscore[align_event]['align_spikes_zscore'][irec][:,:ntest_trials,units_to_use], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
            max_tidx = np.argmax(np.mean(score, axis=1))
            unit_rank = np.flip(np.argsort((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))))
            lda_results[align_event]['scores'].append(score)
            lda_results[align_event]['weights'].append(weights)
            lda_results[align_event]['dec_unit_weight_rank'].append((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))[unit_rank])
        except:
            lda_results[align_event]['scores'].append([])
            lda_results[align_event]['weights'].append([])
            lda_results[align_event]['dec_unit_weight_rank'].append([])
            continue

## Neuron number matched

In [ ]:
# nunits_to_use = np.min([len(stable_unit_idx[irec]) for irec in range(nrecs)])
nunits_to_use = 5
# nunits_to_use = 
for align_event in tqdm(align_events):
    ntime = len(preproc_metadata['trial_time_axis'])
    lda_results[align_event]['neural_space'] = []
    lda_results[align_event]['confusion_matrix'] = []
    for irec in tqdm(range(nrecs)):
        if len(stable_unit_idx[irec]) > 0:
            temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
            temp_lda_neural = np.zeros((ntime-nlda_lags, niter_match))*np.nan
            temp_conf_mat = np.zeros((ntime-nlda_lags, niter_match, len(np.unique(temp_target_labels)), len(np.unique(temp_target_labels))))*np.nan
            for ii in range(niter_match):
                if len(stable_unit_idx[irec]) < nunits_to_use:
                    unit_idx_temp = stable_unit_idx[irec]
                else:
                    unit_idx_temp = np.random.choice(stable_unit_idx[irec], size=nunits_to_use, replace=False)
                temp_da, temp_cm = xval_lda(np.swapaxes(fr_zscore[align_event]['align_spikes_zscore'][irec][:,:,unit_idx_temp], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=nlda_lags, smooth_timeseries=True, return_confusion_matrix=True)
                temp_lda_neural[:,ii] = np.mean(temp_da, axis=1)
                temp_conf_mat[:,ii,:,:] = np.mean(temp_cm, axis=3)

            lda_results[align_event]['neural_space'].append(temp_lda_neural)
            lda_results[align_event]['confusion_matrix'].append(temp_conf_mat)
        else:
            lda_results[align_event]['neural_space'].append([])
            lda_results[align_event]['confusion_matrix'].append([])

In [ ]:
# from multiprocessing import Pool
# import itertools

# start = time.time()
# for irec in tqdm(range(nrecs)):
#     with Pool() as pool:
#         for ii in range(niter_match):
#             data_clusters = [np.swapaxes(fr_zscore[align_event]['align_spikes_zscore'][irec][:,:,np.random.choice(stable_unit_idx[irec], size=nunits_to_use, replace=False)], 1,2) for ii in range(niter_match)]

#         input_args = list(zip(data_clusters, [np_data[align_event]['target_idx_good'][irec] for ii in range(niter_match)], itertools.repeat(1/np_data[align_event]['spike_bin_width_mc']), itertools.repeat(nlda_lags)))
#         test_out = np.array(pool.starmap(xval_lda, input_args))

# print(time.time()-start)

## Pseudopopulations

In [ ]:
nunits_to_use = pseudopopulation_metadata['nrandom_units']   
print(nunits_to_use)
target_idx_list = [np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])]) for irec in range(nrecs)] 
for align_event in align_events:
    lda_results[align_event]['pseudopopulations'] = {}

### Random

In [ ]:
for align_event in align_events:
    lda_results[align_event]['pseudopopulations']['random_scores'] = []
    lda_results[align_event]['pseudopopulations']['random_weights'] = []
    for igroup in tqdm(range(len(random_units))):
        pseudopop_raster, target_labels = get_pseudopopulation_raster(random_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=pseudopopulation_metadata['nrandom_units'])
        temp_scores, temp_weights = xval_lda(np.swapaxes(pseudopop_raster, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=1, smooth_timeseries=True, return_weights=True)
        lda_results[align_event]['pseudopopulations']['random_scores'].append(temp_scores)
        lda_results[align_event]['pseudopopulations']['random_weights'].append(temp_weights)

In [ ]:
# Use only a single time window to compute decoding accuracy
for align_event in align_events:
    lda_results[align_event]['pseudopopulations']['random_scores_event'] = []
    lda_results[align_event]['pseudopopulations']['random_weights_event'] = []
    for igroup in tqdm(range(len(random_units))):
        pseudopop_raster, target_labels = get_pseudopopulation_raster(random_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=pseudopopulation_metadata['nrandom_units'])
        input_data = np.mean(pseudopop_raster[trelevant1:trelevant2,:,:], axis=0).reshape(1,pseudopop_raster.shape[1], pseudopop_raster.shape[2])
        temp_scores, temp_weights = xval_lda(np.swapaxes(input_data, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=1, smooth_timeseries=True, return_weights=True, nfolds=nfolds)
        lda_results[align_event]['pseudopopulations']['random_scores_event'].append(temp_scores)
        lda_results[align_event]['pseudopopulations']['random_weights_event'].append(temp_weights)

### Column

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
lda_results[align_event]['pseudopopulations']['column_scores'] = []
lda_results[align_event]['pseudopopulations']['column_weights'] = []
for igroup in tqdm(range(len(column_units))):
    try:
        temp_scores = np.zeros((ntime, nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
        temp_weights = np.zeros((ntime, ntargets, pseudopopulation_metadata['nrandom_units'],  nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
        for iiter in tqdm(range(pseudopopulation_metadata['nrandom_groups'])):
            pseudopop_raster, target_labels = get_pseudopopulation_raster(column_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=pseudopopulation_metadata['nrandom_units'])
            a, b = xval_lda(np.swapaxes(pseudopop_raster, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
            temp_scores[:,:,iiter] = a
            temp_weights[:,:,:,:,iiter] = b

        lda_results[align_event]['pseudopopulations']['column_scores'].append(np.mean(temp_scores,axis=1))
        lda_results[align_event]['pseudopopulations']['column_weights'].append(np.mean(temp_weights,axis=3))

    except:
        lda_results[align_event]['pseudopopulations']['column_scores'].append([])
        lda_results[align_event]['pseudopopulations']['column_weights'].append([])
        continue

In [ ]:
# Use only a single time window to compute decoding accuracy
# for align_event in align_events:
align_event = align_events[-1]
lda_results[align_event]['pseudopopulations']['column_scores_event'] = []
lda_results[align_event]['pseudopopulations']['column_weights_event'] = []
for igroup in tqdm(range(len(column_units))):
    try:
        temp_scores = np.zeros((ntime, nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
        temp_weights = np.zeros((ntime, ntargets, pseudopopulation_metadata['nrandom_units'],  nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
        for iiter in tqdm(range(pseudopopulation_metadata['nrandom_groups'])):
            pseudopop_raster, target_labels = get_pseudopopulation_raster(column_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=pseudopopulation_metadata['nrandom_units'])
            input_data = np.mean(pseudopop_raster[trelevant1:trelevant2,:,:], axis=0).reshape(1,pseudopop_raster.shape[1], pseudopop_raster.shape[2])
            a, b = xval_lda(np.swapaxes(input_data, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True, nfolds=nfolds)
            temp_scores[:,:,iiter] = a
            temp_weights[:,:,:,:,iiter] = b

        lda_results[align_event]['pseudopopulations']['column_scores_event'].append(np.mean(temp_scores,axis=1))
        lda_results[align_event]['pseudopopulations']['column_weights_event'].append(np.mean(temp_weights,axis=3))

    except:
        print('Exception')
        lda_results[align_event]['pseudopopulations']['column_scores_event'].append([])
        lda_results[align_event]['pseudopopulations']['column_weights_event'].append([])
        continue

### Depth

In [ ]:
# ndepthunits = pseudopopulation_metadata['nrandom_units']
ndepthunits = 5
ntime = len(preproc_metadata['trial_time_axis'])
for align_event in align_events:
    lda_results[align_event]['pseudopopulations']['depth_scores'] = []
    lda_results[align_event]['pseudopopulations']['depth_weights'] = []
    for igroup in tqdm(range(len(depth_units))):
        temp_scores = np.zeros((ntime, nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
        temp_weights = np.zeros((ntime, ntargets, ndepthunits,  nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
        for iiter in range(pseudopopulation_metadata['nrandom_groups']):
            pseudopop_raster, target_labels = get_pseudopopulation_raster(depth_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=ndepthunits)
            a, b = xval_lda(np.swapaxes(pseudopop_raster, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
            temp_scores[:,:,iiter] = a
            temp_weights[:,:,:,:,iiter] = b
        
        lda_results[align_event]['pseudopopulations']['depth_scores'].append(np.mean(temp_scores,axis=1))
        lda_results[align_event]['pseudopopulations']['depth_weights'].append(np.mean(temp_weights,axis=3))


In [ ]:
# Plot how many units are in each recording site at each depth
# niterations = pseudopopulation_metadata['nrandom_groups']
ndepthunits = 5
niterations = 50
ntime = len(preproc_metadata['trial_time_axis'])
target_idx_list = [np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])]) for irec in range(nrecs)] 
align_event = align_events[-1]
# for align_event in align_events:
lda_results[align_event]['pseudopopulations']['depth_scores_by_site'] = {}
lda_results[align_event]['pseudopopulations']['depth_weights_by_site'] = {}
for isite, site in enumerate(tqdm(list(depth_group_info_by_site.keys()))):
    lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site] = []
    lda_results[align_event]['pseudopopulations']['depth_weights_by_site'][site] = []
    for igroup in range(len(depth_group_info_by_site[site])):
        temp_scores = np.zeros((ntime, nfolds, niterations))*np.nan
        temp_weights = np.zeros((ntime, ntargets, ndepthunits,  nfolds, niterations))*np.nan
        if len(depth_group_info_by_site[site][igroup]) >= ndepthunits:
            for iiter in range(niterations):
                pseudopop_raster, target_labels = get_pseudopopulation_raster(depth_group_info_by_site[site][igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=ndepthunits)
                a, b = xval_lda(np.swapaxes(pseudopop_raster, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
                temp_scores[:,:,iiter] = a
                temp_weights[:,:,:,:,iiter] = b

        lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site].append(np.mean(temp_scores,axis=1))
        lda_results[align_event]['pseudopopulations']['depth_weights_by_site'][site].append(np.mean(temp_weights,axis=3))

### High decoding units

In [ ]:
# Extract df for high decoding units
# nimportant_units = 50
nimportant_units = 20
# nimportant_units = pseudopopulation_metadata['nrandom_units']
high_decoding_units = []
low_decoding_units = []
all_sorted_weights = []
for ievent, event in enumerate(align_events):
    sorted_weights = np.argsort(np.sum(np.max(np.abs(np.mean(lda_results[align_event]['all_units_weight'], axis=3))[trelevant1:trelevant2,:,:], axis=1), axis=0))
    all_sorted_weights.append(sorted_weights)
    important_units = sorted_weights[-nimportant_units:]
    unimportant_units = sorted_weights[:nimportant_units]
    high_decoding_units.append(unit_df.iloc[important_units].reset_index())
    low_decoding_units.append(unit_df.iloc[unimportant_units].reset_index())

In [ ]:
# Calculate decoding accuracy for each
for ievent, align_event in enumerate(align_events):
    lda_results[align_event]['pseudopopulations']['high_dec_scores'] = []
    lda_results[align_event]['pseudopopulations']['high_dec_weights'] = []
    pseudopop_raster, target_labels = get_pseudopopulation_raster(high_decoding_units[ievent].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=nimportant_units)
    temp_scores, temp_weights = xval_lda(np.swapaxes(pseudopop_raster, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
    
    lda_results[align_event]['pseudopopulations']['high_dec_scores'] = np.mean(temp_scores,axis=1)
    lda_results[align_event]['pseudopopulations']['high_dec_weights'] = np.mean(temp_weights,axis=3)

## Trial adding

In [ ]:
# lda_results_tradd = {}
# for align_event in align_events:
#     lda_results_tradd[align_event] = {}
#     lda_results_tradd[align_event]['neural_space'] = []
#     nunits_to_use_tradd = np.min([np.min([len(stable_unit_idx_nunits_tradd[align_event][itradd_bin][irec]) for irec in range(nrecs)]) for itradd_bin in range(ntrial_adding_bins)])
#     for itradd_bin in range(ntrial_adding_bins):
#         lda_neural_tradd_temp = []

#         for irec in tqdm(range(nrecs)):
#             temp_lda_neural = np.zeros((ntime-nlda_lags, niter_match))*np.nan
#             for ii in range(niter_match):
#                 unit_idx_temp = np.random.choice(stable_unit_idx_nunits_tradd[align_event][itradd_bin][irec], size=nunits_to_use_tradd, replace=False)
#                 temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
#                 try:
#                     temp_lda_neural[:,ii] = np.mean(xval_lda(np.swapaxes(fr_zscore_tradd[align_event]['align_spikes_zscore'][itradd_bin][irec][:,:,unit_idx_temp], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=nlda_lags, smooth_timeseries=False), axis=1)
#                 except:
#                     print('Could not perform LDA')
#                     continue
                    
#             lda_neural_tradd_temp.append(temp_lda_neural)
#         lda_results_tradd[align_event]['neural_space'].append(lda_neural_tradd_temp)

## Unit adding

In [ ]:
groups_of_nunits = np.arange(1, 30, 1)
# for align_event in align_events:
align_event = align_events[-1]
lda_results[align_event]['unit_adding']['scores_recording'] = {}
# lda_results[align_event]['pseudopopulations']['column_weights'] = []
for igroup in tqdm(range(len(column_units))):
    lda_results[align_event]['unit_adding']['scores_recording'][igroup] = []
    for group_size in groups_of_nunits:
        if group_size <= len(column_units[igroup].reset_index()):
            temp_scores = np.zeros((ntime-1, nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
            temp_weights = np.zeros((ntime-1, ntargets, group_size,  nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
            for iiter in range(pseudopopulation_metadata['nrandom_groups']):
                pseudopop_raster, target_labels = get_pseudopopulation_raster(column_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=group_size)
                a, b = xval_lda(np.swapaxes(pseudopop_raster, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=1, smooth_timeseries=True, return_weights=True)
                temp_scores[:,:,iiter] = a
                # temp_weights[:,:,:,:,iiter] = b

            lda_results[align_event]['unit_adding']['scores_recording'][igroup].append(np.mean(temp_scores,axis=1))
        else:
            continue

        # lda_results[align_event]['pseudopopulations']['column_weights'].append(np.mean(temp_weights,axis=3))

## Spike band power decoding

In [ ]:
# input_data = np.mean(pseudopop_raster[trelevant1:trelevant2,:,:], axis=0).reshape(1,pseudopop_raster.shape[1], pseudopop_raster.shape[2])
print(input_data.shape)
np.concatenate(all_sbp_data_list, axis=2).shape
all_sbp_data_list[0].shape

In [ ]:
# Organize data input to xval_lda
for align_event in tqdm(align_events):
    all_sbp_data_list = []
    for irec in tqdm(range(nrecs)):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        sorted_target_label_mask = np.argsort(temp_target_labels)
        all_sbp_data_list.append(fr_zscore[align_event]['align_sbp_zscore'][irec][:,sorted_target_label_mask,:][:,:,:])

    lda_results[align_event]['sbp']['all_units_score'], lda_results[align_event]['sbp']['all_units_weight'] = xval_lda(np.swapaxes(np.concatenate(all_sbp_data_list, axis=2), 1,2), temp_target_labels[sorted_target_label_mask], 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)

In [ ]:
# Calculate decoding accuracy for each recording to get ranked order of weights
ntest_trials = fr_zscore[align_event]['align_sbp_zscore'][0].shape[1]
for align_event in align_events:
    lda_results[align_event]['sbp']['scores'], lda_results[align_event]['sbp']['weights'] = [], []
    lda_results[align_event]['sbp']['unit_rank'], lda_results[align_event]['sbp']['dec_unit_weight_rank'] = [], []
    for irec in tqdm(range(nrecs)):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        score, weights = xval_lda(np.swapaxes(fr_zscore[align_event]['align_sbp_zscore'][irec][:,:ntest_trials,:], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        max_tidx = np.argmax(np.mean(score, axis=1))
        unit_rank = np.flip(np.argsort((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))))
        lda_results[align_event]['sbp']['scores'].append(score)
        lda_results[align_event]['sbp']['weights'].append(weights)
        lda_results[align_event]['sbp']['unit_rank'].append(unit_rank)
        lda_results[align_event]['sbp']['dec_unit_weight_rank'].append((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))[unit_rank])

In [ ]:
# Use on only a time window around the event
# Calculate decoding accuracy for each recording to get ranked order of weights
ntest_trials = fr_zscore[align_event]['align_sbp_zscore'][0].shape[1]
for align_event in align_events:
    lda_results[align_event]['sbp']['scores_event'], lda_results[align_event]['sbp']['weights_event'] = [], []
    lda_results[align_event]['sbp']['unit_rank_event'], lda_results[align_event]['sbp']['dec_unit_weight_rank_event'] = [], []
    for irec in tqdm(range(nrecs)):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        input_data = np.mean(fr_zscore[align_event]['align_sbp_zscore'][irec][trelevant1:trelevant2,:,:], axis=0).reshape(1,fr_zscore[align_event]['align_sbp_zscore'][irec].shape[1], fr_zscore[align_event]['align_sbp_zscore'][irec].shape[2])
        score, weights = xval_lda(np.swapaxes(input_data[:,:ntest_trials,:], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        max_tidx = np.argmax(np.mean(score, axis=1))
        unit_rank = np.flip(np.argsort((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))))
        lda_results[align_event]['sbp']['scores_event'].append(score)
        lda_results[align_event]['sbp']['weights_event'].append(weights)
        lda_results[align_event]['sbp']['unit_rank_event'].append(unit_rank)
        lda_results[align_event]['sbp']['dec_unit_weight_rank_event'].append((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))[unit_rank])

In [ ]:
# Take the not n weighted channels and performed decoding with them
nsbp_ch = 50
for align_event in align_events:
    lda_results[align_event]['sbp']['scores_top_ch'], lda_results[align_event]['sbp']['weights_top_ch'] = [], []
    for irec in tqdm(range(nrecs)):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        chs_use = lda_results[align_event]['sbp']['unit_rank'][irec][:nsbp_ch]
        score, weights = xval_lda(np.swapaxes(fr_zscore[align_event]['align_sbp_zscore'][irec][:,:ntest_trials,chs_use], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        lda_results[align_event]['sbp']['scores_top_ch'].append(score)
        lda_results[align_event]['sbp']['weights_top_ch'].append(weights)

In [ ]:
# Use on only a time window around the event
# Take the not n weighted channels and performed decoding with them
nsbp_ch = 50
for align_event in align_events:
    lda_results[align_event]['sbp']['scores_top_ch_event'], lda_results[align_event]['sbp']['weights_top_ch_evemt'] = [], []
    for irec in tqdm(range(nrecs)):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        chs_use = lda_results[align_event]['sbp']['unit_rank_event'][irec][:nsbp_ch]
        input_data = np.mean(fr_zscore[align_event]['align_sbp_zscore'][irec][trelevant1:trelevant2,:,:], axis=0).reshape(1,fr_zscore[align_event]['align_sbp_zscore'][irec].shape[1], fr_zscore[align_event]['align_sbp_zscore'][irec].shape[2])
        score, weights = xval_lda(np.swapaxes(input_data[:,:ntest_trials,chs_use], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        lda_results[align_event]['sbp']['scores_top_ch_event'].append(score)
        lda_results[align_event]['sbp']['weights_top_ch_evemt'].append(weights)

In [ ]:
 # Single channel decoding
# for align_event in align_events:
align_event = align_events[-1]
lda_results[align_event]['sbp']['single_ch_decoding'] = []
for irec in tqdm(range(nrecs)):
    temp_single_ch_decoding = np.zeros((ntime,fr_zscore[align_event]['align_sbp_zscore'][irec].shape[2],nfolds))*np.nan
    for ich in range(fr_zscore[align_event]['align_sbp_zscore'][irec].shape[2]):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        score, _ = xval_lda(np.swapaxes(fr_zscore[align_event]['align_sbp_zscore'][irec][:,:,ich][:,:,None], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        temp_single_ch_decoding[:,ich,:] = score
    lda_results[align_event]['sbp']['single_ch_decoding'].append(temp_single_ch_decoding)

In [ ]:
# Use on only a time window around the event 
# Single channel decoding
# for align_event in align_events:
align_event = align_events[-1]
lda_results[align_event]['sbp']['single_ch_decoding_event'] = []
for irec in tqdm(range(nrecs)):
    temp_single_ch_decoding = np.zeros((ntime,fr_zscore[align_event]['align_sbp_zscore'][irec].shape[2],nfolds))*np.nan
    input_data = np.mean(fr_zscore[align_event]['align_sbp_zscore'][irec][trelevant1:trelevant2,:,:], axis=0).reshape(1,fr_zscore[align_event]['align_sbp_zscore'][irec].shape[1], fr_zscore[align_event]['align_sbp_zscore'][irec].shape[2])

    for ich in range(fr_zscore[align_event]['align_sbp_zscore'][irec].shape[2]):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        score, _ = xval_lda(np.swapaxes(input_data[:,:,ich][:,:,None], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        temp_single_ch_decoding[:,ich,:] = score
    lda_results[align_event]['sbp']['single_ch_decoding_event'].append(temp_single_ch_decoding)

## LFP Power decoding 

In [ ]:
# Organize data input to xval_lda
align_event = align_events[-1]
for iband in tqdm(range(nbands)):
    all_lfp_data_list = []
    for irec in range(nrecs):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        sorted_target_label_mask = np.argsort(temp_target_labels)
        all_lfp_data_list.append(fr_zscore[align_event]['align_lfp_zscore'][iband][irec][:,sorted_target_label_mask,:])

    lda_results[align_event]['lfp'][iband]['all_units_score'], lda_results[align_event]['lfp'][iband]['all_units_weight'] = xval_lda(np.swapaxes(np.concatenate(all_lfp_data_list, axis=2), 1,2), temp_target_labels[sorted_target_label_mask], 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)

In [ ]:
# Calculate decoding accuracy for each recording to get ranked order of weights
ntest_trials = fr_zscore[align_event]['align_lfp_zscore'][0][0].shape[1]
align_event = align_events[-1]
for iband in tqdm(range(nbands)):
    lda_results[align_event]['lfp'][iband] = {}
    lda_results[align_event]['lfp'][iband]['scores'], lda_results[align_event]['lfp'][iband]['weights'] = [], []
    lda_results[align_event]['lfp'][iband]['unit_rank'], lda_results[align_event]['lfp'][iband]['dec_unit_weight_rank'] = [], []
    for irec in tqdm(range(nrecs)):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        score, weights = xval_lda(np.swapaxes(fr_zscore[align_event]['align_lfp_zscore'][iband][irec][:,:ntest_trials,:], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
        max_tidx = np.argmax(np.mean(score, axis=1))
        unit_rank = np.flip(np.argsort((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))))
        lda_results[align_event]['lfp'][iband]['scores'].append(score)
        lda_results[align_event]['lfp'][iband]['weights'].append(weights)
        lda_results[align_event]['lfp'][iband]['unit_rank'].append(unit_rank)
        lda_results[align_event]['lfp'][iband]['dec_unit_weight_rank'].append((np.sum(np.abs(np.mean(weights, axis=3)[max_tidx,:,:]), axis=0))[unit_rank])

In [ ]:
# LFP single channel decoding
ntest_trials = fr_zscore[align_event]['align_lfp_zscore'][0][0].shape[1]
align_event = align_events[-1]
lda_results[align_event]['lfp_single_ch'] = {}
for iband in tqdm(range(nbands)):
    lda_results[align_event]['lfp_single_ch'][iband] = {}
    lda_results[align_event]['lfp_single_ch'][iband]['scores'] = []
    for irec in tqdm(range(nrecs)):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['date']==dates[irec])])
        temp_single_ch_decoding = np.zeros((ntime,fr_zscore[align_event]['align_lfp_zscore'][iband][irec].shape[2],nfolds))*np.nan
        for ich in range(fr_zscore[align_event]['align_lfp_zscore'][iband][irec].shape[2]):
            score, weights = xval_lda(np.swapaxes(fr_zscore[align_event]['align_lfp_zscore'][iband][irec][:,:,ich][:,:,None], 1,2), temp_target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=True, return_weights=True)
            temp_single_ch_decoding[:,ich,:] = score
        lda_results[align_event]['lfp_single_ch'][iband]['scores'].append(temp_single_ch_decoding)


# Calculate linear regression decoding accuracy

## Functions

In [ ]:
def calc_task_rel_dims(neural_data, kin_data, conc_proj_data=False, regularization=None, alpha=1):
    '''
    Calculates the task relevant dimensions by regressing neural activity against kinematic data using least squares.
    If the input neural data is 3D, all trials will be concatenated to calculate the subspace. 
    Calculation is based on the approach used in Sun et al. 2022 https://doi.org/10.1038/s41586-021-04329-x
    
    .. math::
    
        R \\in \\mathbb{R}^{nt \\times nch}
        M \\in \\mathbb{R}^{nt \\times nkin}
        \\beta \\in \\mathbb{R}^{nch \\times nkin}
        R = M\\beta^T
        [\\beta_0 \beta_x \beta_y]^T = (M^T M)^{-1} M^T R

    Args:
        neural_data ((nt, nch) or list of (nt, nch)): Input neural data (:math:`R`) to regress against kinematic activity.
        kin_data ((nt, ndim) or list of (nt, ndim)): Kinematic variables (:math:`M`), commonly position or instantaneous velocity. 'ndims' refers to the number of physical dimensions that define the kinematic data (i.e. X and Y)
        conc_proj_data (bool): If the projected neural data should be concatenated.

    Returns:
        tuple: Tuple containing:
            | **(nch, ndim):** Subspace (:math:`\beta`) that best predicts kinematic variables. Note the first column represents the intercept, then the next dimensions represent the behvaioral variables
            | **((nt, nch) or list of (nt, ndim)):** Neural data projected onto task relevant subspace

    '''

    # If a list of segments from trials, concatenate them into one larget timeseries
    if type(neural_data) == list:
        ntrials = len(neural_data)
        min_time_per_trial = [np.min([len(neural_data[itrial]), len(kin_data[itrial])]) for itrial in range(ntrials)]
        
        conc_neural_data = np.vstack([neural_data[itrial][:min_time_per_trial[itrial]] for itrial in range(ntrials)]) #(nt, nch)
        conc_kin_data_baseline = np.vstack([kin_data[itrial][:min_time_per_trial[itrial]] for itrial in range(ntrials)])
        if conc_neural_data.shape[0] != conc_kin_data_baseline.shape[0]:
            print(f"Neural and kinematic data sizes are {np.abs(conc_neural_data.shape[0]-conc_kin_data_baseline.shape[0])} samples apart")
        
        ntime = np.min([conc_neural_data.shape[0], conc_kin_data_baseline.shape[0]])
        
        # Set input neural data as a float
        conc_neural_data = conc_neural_data.astype(float)[:ntime,:]

        conc_kin_data = np.ones((ntime,kin_data[0].shape[1]+1))*np.nan
        conc_kin_data[:,0] = 1
        conc_kin_data[:,1:] = conc_kin_data_baseline

        # Center neural data:
        conc_neural_data -= np.nanmean(conc_neural_data, axis=0)

        # Calculate task relevant subspace 
        # task_subspace = np.linalg.pinv(conc_kin_data.T @ conc_kin_data) @ conc_kin_data.T @ conc_neural_data
        if regularization is None:        
            # task_subspace = np.linalg.pinv(conc_neural_data.T @ conc_neural_data) @ conc_neural_data.T @ conc_kin_data
            lin_reg_model = sklearn.linear_model.LinearRegression(fit_intercept=False).fit(conc_neural_data, conc_kin_data)
            task_subspace = (lin_reg_model.coef_).T
        elif regularization == 'lasso':
            lasso_model = sklearn.linear_model.Lasso(alpha=alpha, max_iter=10000).fit(conc_neural_data, conc_kin_data)
            task_subspace = (lasso_model.coef_).T
        elif regularization == 'ridge':
            ridge_model = sklearn.linear_model.Ridge(alpha=alpha, max_iter=10000).fit(conc_neural_data, conc_kin_data)
            task_subspace = (ridge_model.coef_).T
        elif regularization == 'elastic net':
            elasticnet_model = sklearn.linear_model.ElasticNet(alpha=alpha, max_iter=10000).fit(conc_neural_data, conc_kin_data)
            task_subspace = (elasticnet_model.coef_).T
    
    else:
        # Save original neural data as a list
        neural_data = [neural_data]
        
        # Set input neural data as a float
        neural_data_centered = neural_data[0].astype(float)
        
        # Center neural data:
        neural_data_centered -= np.nanmean(neural_data_centered, axis=0)
        ntime = neural_data_centered.shape[0]
        conc_kin_data = np.ones((ntime, kin_data.shape[1]+1))*np.nan
        conc_kin_data[:,0] = 1
        conc_kin_data[:,1:] = kin_data
        
        # Calculate task relevant subspace 
        if regularization is None:
            task_subspace = np.linalg.pinv(neural_data_centered.T @ neural_data_centered) @ neural_data_centered.T @ conc_kin_data
        elif regularization == 'lasso':
            lasso_model = sklearn.linear_model.Lasso(alpha=alpha, max_iter=10000).fit(neural_data_centered, conc_kin_data)
            task_subspace = (lasso_model.coef_).T
        elif regularization == 'ridge':
            ridge_model = sklearn.linear_model.Ridge(alpha=alpha, max_iter=10000).fit(neural_data_centered, conc_kin_data)
            task_subspace = (ridge_model.coef_).T
        elif regularization == 'elastic net':
            elasticnet_model = sklearn.linear_model.ElasticNet(alpha=alpha, max_iter=10000).fit(neural_data_centered, conc_kin_data)
            task_subspace = (elasticnet_model.coef_).T
        ntrials = 1
        
    # Project neural data onto task subspace
    projected_data = []
    
    for itrial in range(ntrials):
        projected_data.append(neural_data[itrial] @ task_subspace)

    if conc_proj_data:
        return task_subspace, np.vstack(projected_data)
    else:    
        return task_subspace, projected_data
    
def xval_task_rel_dims(neural_data, kin_data, labels=None, regularization=None, nfolds=4, alpha=1, smooth_neural_data=True, tavg_neural_data=False, return_test_label_idx=False):
    '''
    '''    
    test_pred_move = []
    test_move = []
    test_label_idx = []
    scores = np.zeros((nfolds,2))*np.nan # Score for each fold and separately for x and y
    if isinstance(neural_data, np.ndarray):
        ntrials = neural_data.shape[1]
        nunits = neural_data.shape[2]
        nkindim = kin_data.shape[2]
    else:
        ntrials = len(neural_data)
        nunits = neural_data[0].shape[1]
        nkindim = kin_data[0].shape[1]
    kf = sklearn.model_selection.KFold(n_splits=nfolds,shuffle=True,random_state=1)
    
    subspaces = np.zeros((nunits, nkindim, nfolds))*np.nan
    
    for ifold, (train_idx, test_idx) in enumerate(kf.split(np.ones((ntrials,ntrials)))):
        if tavg_neural_data:
            train_neural_data = avg_by_condition(neural_data[:,train_idx,:], labels[train_idx])
            test_neural_data = np.mean(neural_data[:,test_idx,:], axis=0)
            train_kin_data = avg_by_condition(kin_data[:,train_idx,:], labels[train_idx])
            test_kin_data = np.mean(kin_data[:,test_idx,:], axis=0)
            
            # Center test neural data
            # test_neural_data -= np.mean(test_neural_data, axis=1)[:,None]
            test_neural_data -= np.mean(test_neural_data, axis=0)
                                          
        else:
            # Train
            train_neural_data = [smooth_timeseries_gaus(neural_data[itrain_idx], samplerate=100, width=250, nstd=3) for itrain_idx in train_idx]
            test_neural_data = np.vstack([smooth_timeseries_gaus(neural_data[itest_idx], samplerate=100, width=250, nstd=3) for itest_idx in test_idx])
            train_kin_data = [kin_data[itrain_idx] for itrain_idx in train_idx]
            test_kin_data = [kin_data[itest_idx] for itest_idx in test_idx]

            # Center test neural data
            test_neural_data -= np.nanmean(test_neural_data, axis=0)
            
        # print(np.vstack(train_neural_data).shape, np.vstack(train_kin_data).shape)
        train_subspace,_  = calc_task_rel_dims(train_neural_data, train_kin_data, conc_proj_data=False, regularization=regularization, alpha=alpha)
    
        # Test
        pred_move = test_neural_data @ train_subspace
        # pred_move = np.linalg.pinv(train_subspace) @ test_neural_data.T
        
        ntime = np.min([np.vstack(test_kin_data)[:,0].shape[0], pred_move.shape[0]])
        # print(test_kin_data.shape, np.vstack(test_kin_data).shape)
        # Score
        scores[ifold,0] = sklearn.metrics.r2_score(np.vstack(test_kin_data)[:ntime,0], pred_move[:ntime,1]) # Score x component
        scores[ifold,1] = sklearn.metrics.r2_score(np.vstack(test_kin_data)[:ntime,1], pred_move[:ntime,2]) # Score y component
        
        # Record results
        if return_test_label_idx:
            test_label_idx.append(labels[test_idx])
        test_pred_move.append(pred_move)
        if tavg_neural_data:
            test_move.append(test_kin_data)
        else:
            test_move.append(np.vstack(test_kin_data))
        subspaces[:,:,ifold] = train_subspace[:,1:] # first subspace is the intercept
        
        
    if return_test_label_idx:
        return scores, test_move, test_pred_move, subspaces, test_label_idx
    else:
        return scores, test_move, test_pred_move, subspaces

In [ ]:
def get_spike_seg_array(df, align_idx_string, samplerate, smooth=True):

    unique_unit_labels = np.sort(np.unique([list(itrial_data.keys()) for itrial_data in list(df['spike_segs'])]).astype(int))   
    ntrials = len(df)
    raster_list = []
    
    for itrial in range(ntrials):
        start_idx = int(list(df[align_idx_string])[itrial])
        ntime = int(len(list(df['spike_segs'])[itrial][str(unique_unit_labels[0])])) - start_idx
        nunits = len(unique_unit_labels)
        raster_array = np.zeros((ntime, nunits))*np.nan
        
        for iunit, unit_label in enumerate(unique_unit_labels):
            if smooth:
                raster_array[:,iunit] = smooth_timeseries_gaus(list(df['spike_segs'])[itrial][str(unit_label)], samplerate, width=150, nstd=3)[start_idx:]
            else:
                raster_array[:,iunit] = list(df['spike_segs'])[itrial][str(unit_label)][start_idx:]
                
        raster_list.append(raster_array)
    return raster_list

## All units

In [ ]:
lin_reg_results = {}
for align_event in align_events:
    lin_reg_results[align_event] = {}

### Trial averaged per target

In [ ]:
def avg_by_condition(data, cond_labels):
    '''
    args:
        data (ntime, ntrials, nch)
        labels (ntrials)
        
    returns
        (ncond, nch)
    '''
    ntime, ntrials, nch = data.shape
    unique_labels = np.unique(cond_labels)
    avg_data = np.zeros((len(unique_labels), nch))*np.nan
    for ilabel, label in enumerate(unique_labels):       
        avg_data[ilabel,:] = np.mean(data[:,cond_labels==label,:],axis=(0,1))
        
    return avg_data

In [ ]:
tavg_time = .5 #[s]
neural_onset_idx_labels = ['delay_start_neural_idx', 'go_cue_neural_idx', 'mov_onset_neural_idx']
kin_onset_idx_labels = ['delay_start_kin_idx', 'go_cue_kin_idx', 'mov_onset_kin_idx']

for ievent, align_event in enumerate(align_events):
    lin_reg_results[align_event]['scores_tavg'] = []
    lin_reg_results[align_event]['test_velo_tavg'] = []
    lin_reg_results[align_event]['test_pred_velo_tavg'] = []
    lin_reg_results[align_event]['subspaces_tavg'] = []
    lin_reg_results[align_event]['test_label_idx_tavg'] = []
    lin_reg_results[align_event]['scores_tavg_hand'] = []
    lin_reg_results[align_event]['test_velo_tavg_hand'] = []
    lin_reg_results[align_event]['test_pred_velo_tavg_hand'] = []
    lin_reg_results[align_event]['subspaces_tavg_hand'] = []
    lin_reg_results[align_event]['test_label_idx_tavg_hand'] = []
    for idate, date in enumerate(tqdm(dates)):
        neural_idx_start = int(preproc_metadata['tbefore']*preproc_metadata['neural_samplerate'])
        neural_idx_stop = int((tavg_time*preproc_metadata['neural_samplerate'])) + neural_idx_start        
        neural_data = rasters['neural'][align_event][idate][neural_idx_start:neural_idx_stop,np.array(df['good_trial'][df['date']==date])][:,:,stable_unit_idx[idate]]
        
        kin_idx_start = int(preproc_metadata['tbefore']*preproc_metadata['kin_samplerate'])
        kin_idx_stop = int((tavg_time*preproc_metadata['kin_samplerate'])) + kin_idx_start
        kin_data = rasters['cursor_velo'][align_event][idate][kin_idx_start:kin_idx_stop,np.array(df['good_trial'][df['date']==date])]
        hand_data = rasters['hand_velo'][align_event][idate][kin_idx_start:kin_idx_stop,np.array(df['good_trial'][df['date']==date])]
        
        scores, test_move_tavg, test_pred_move_tavg, subspace, test_label_idx =  xval_task_rel_dims(neural_data, kin_data, labels=np.array(df['target_idx'][(df['date']==date)*df['good_trial']])  ,regularization=None, nfolds=4, alpha=1, smooth_neural_data=False, tavg_neural_data=True, return_test_label_idx=True)
        scores_hand, test_move_tavg_hand, test_pred_move_tavg_hand, subspace_hand, test_label_idx_hand =  xval_task_rel_dims(neural_data, hand_data, labels=np.array(df['target_idx'][(df['date']==date)*df['good_trial']])  ,regularization=None, nfolds=4, alpha=1, smooth_neural_data=False, tavg_neural_data=True, return_test_label_idx=True)
        
        lin_reg_results[align_event]['scores_tavg'].append(scores)
        lin_reg_results[align_event]['test_velo_tavg'].append(test_move_tavg)
        lin_reg_results[align_event]['test_pred_velo_tavg'].append(test_pred_move_tavg)
        lin_reg_results[align_event]['subspaces_tavg'].append(subspace)
        lin_reg_results[align_event]['test_label_idx_tavg'].append(test_label_idx)
        lin_reg_results[align_event]['scores_tavg_hand'].append(scores_hand)
        lin_reg_results[align_event]['test_velo_tavg_hand'].append(test_move_tavg_hand)
        lin_reg_results[align_event]['test_pred_velo_tavg_hand'].append(test_pred_move_tavg_hand)
        lin_reg_results[align_event]['subspaces_tavg_hand'].append(subspace_hand)
        lin_reg_results[align_event]['test_label_idx_tavg_hand'].append(test_label_idx_hand)

### Concatenated trials

In [ ]:
# look at movement onset regressed against cursor position
neural_onset_idx_labels = ['delay_start_neural_idx', 'go_cue_neural_idx', 'mov_onset_neural_idx']
kin_onset_idx_labels = ['delay_start_kin_idx', 'go_cue_kin_idx', 'mov_onset_kin_idx']
for ievent, align_event in enumerate(align_events):
    lin_reg_results[align_event]['scores_pos'] = []
    lin_reg_results[align_event]['test_pos'] = []
    lin_reg_results[align_event]['test_pred_pos'] = []
    lin_reg_results[align_event]['subspaces_pos'] = []
    for idate, date in enumerate(tqdm(dates)):
        neural_data = get_spike_seg_array(df[df['date']==date], neural_onset_idx_labels[ievent], samplerate=100, smooth=True)
        kin_data = [cursor_seg[int(list(df[kin_onset_idx_labels[ievent]][df['date']==date])[iseg]):] for iseg, cursor_seg in enumerate(df['cursor_traj'][df['date']==date]) if np.array(df['good_trial'][df['date']==date])[iseg]]
        stable_neural_data = [neural_data[itrial][:,stable_unit_idx[idate]] for itrial in range(len(neural_data)) if np.array(df['good_trial'][df['date']==date])[itrial]]
        
        # Down sample kinematic data to match neural data
        if preproc_metadata['kin_samplerate'] != preproc_metadata['neural_samplerate']:
            dwns_kin_data = [aopy.precondition.base.downsample(kin_data[iseg], preproc_metadata['kin_samplerate'], preproc_metadata['neural_samplerate']) for iseg in range(len(kin_data))]
        
        scores, test_move, test_pred_move, subspaces = xval_task_rel_dims(stable_neural_data, dwns_kin_data,regularization=None, nfolds=4)
        lin_reg_results[align_event]['scores_pos'].append(scores)
        lin_reg_results[align_event]['test_pos'].append(test_move)
        lin_reg_results[align_event]['test_pred_pos'].append(test_pred_move)
        lin_reg_results[align_event]['subspaces_pos'].append(subspaces)

In [ ]:
# look at movement onset regressed against cursor velo
neural_onset_idx_labels = ['delay_start_neural_idx', 'go_cue_neural_idx', 'mov_onset_neural_idx']
kin_onset_idx_labels = ['delay_start_kin_idx', 'go_cue_kin_idx', 'mov_onset_kin_idx']
for ievent, align_event in enumerate(align_events):
    lin_reg_results[align_event]['scores_velo'] = []
    lin_reg_results[align_event]['test_velo'] = []
    lin_reg_results[align_event]['test_pred_velo'] = []
    lin_reg_results[align_event]['subspaces_velo'] = []
    for idate, date in enumerate(tqdm(dates)):
        neural_data = get_spike_seg_array(df[df['date']==date], neural_onset_idx_labels[ievent], samplerate=100, smooth=True)
        stable_neural_data = [neural_data[itrial][:,stable_unit_idx[idate]] for itrial in range(len(neural_data)) if np.array(df['good_trial'][df['date']==date])[itrial]]
        
        kin_data = [cursor_seg[int(list(df[kin_onset_idx_labels[ievent]][df['date']==date])[iseg]):] for iseg, cursor_seg in enumerate(df['cursor_vel_traj'][df['date']==date])  if np.array(df['good_trial'][df['date']==date])[iseg]]
        if preproc_metadata['kin_samplerate'] != preproc_metadata['neural_samplerate']:
            dwns_kin_data = [aopy.precondition.base.downsample(kin_data[iseg], preproc_metadata['kin_samplerate'], preproc_metadata['neural_samplerate']) for iseg in range(len(kin_data))]
        
        scores, test_move, test_pred_move, subspaces = xval_task_rel_dims(stable_neural_data, dwns_kin_data,regularization='ridge', nfolds=4)
        lin_reg_results[align_event]['scores_velo'].append(scores)
        lin_reg_results[align_event]['test_velo'].append(test_move)
        lin_reg_results[align_event]['test_pred_velo'].append(test_pred_move)
        lin_reg_results[align_event]['subspaces_velo'].append(subspaces)

## Neuron number matched

In [ ]:
tavg_time = .5 #[s]
neural_onset_idx_labels = ['delay_start_neural_idx', 'go_cue_neural_idx', 'mov_onset_neural_idx']
kin_onset_idx_labels = ['delay_start_kin_idx', 'go_cue_kin_idx', 'mov_onset_kin_idx']
nunits_to_use = np.min([len(stable_unit_idx[irec]) for irec in range(nrecs)])
for ievent, align_event in enumerate(align_events):
    lin_reg_results[align_event]['mae_tavg_nmatch'] = []
    # lin_reg_results[align_event]['test_velo_tavg'] = []
    # lin_reg_results[align_event]['test_pred_velo_tavg'] = []
    # lin_reg_results[align_event]['subspaces_tavg'] = []
    # lin_reg_results[align_event]['test_label_idx_tavg'] = []
    for idate, date in enumerate(tqdm(dates)):
        
        temp_lin_reg_scores = np.zeros((nfolds, 2, niter_match))*np.nan
        mae = np.zeros(niter_match)*np.nan
        for ii in range(niter_match):
            unit_idx_temp = np.random.choice(stable_unit_idx[idate], size=nunits_to_use, replace=False)
            
            neural_idx_start = int(preproc_metadata['tbefore']*preproc_metadata['neural_samplerate'])
            neural_idx_stop = int((tavg_time*preproc_metadata['neural_samplerate'])) + neural_idx_start        
            neural_data = rasters['neural'][align_event][idate][neural_idx_start:neural_idx_stop,np.array(df['good_trial'][df['date']==date])][:,:,unit_idx_temp]

            kin_idx_start = int(preproc_metadata['tbefore']*preproc_metadata['kin_samplerate'])
            kin_idx_stop = int((tavg_time*preproc_metadata['kin_samplerate'])) + kin_idx_start
            kin_data = rasters['cursor_velo'][align_event][idate][kin_idx_start:kin_idx_stop,np.array(df['good_trial'][df['date']==date])]

            scores, test_move_tavg, test_pred_move_tavg, subspace, test_label_idx =  xval_task_rel_dims(neural_data, kin_data, labels=np.array(df['target_idx'][(df['date']==date)*df['good_trial']])  ,regularization=None, nfolds=nfolds, alpha=1, smooth_neural_data=False, tavg_neural_data=True, return_test_label_idx=True)
            temp_lin_reg_scores[:,:,ii] = scores
            
            mae_x = np.mean([sklearn.metrics.mean_absolute_error(test_move_tavg[ifold][:,0],test_pred_move_tavg[ifold][:,1]) for ifold in range(nfolds)])
            mae_y = np.mean([sklearn.metrics.mean_absolute_error(test_move_tavg[ifold][:,1], test_pred_move_tavg[ifold][:,2]) for ifold in range(nfolds)])
            mae[ii] = np.sqrt(mae_x**2+mae_y**2)
        
        lin_reg_results[align_event]['mae_tavg_nmatch'].append(mae)
        # lin_reg_results[align_event]['test_velo_tavg'].append(test_move_tavg)
        # lin_reg_results[align_event]['test_pred_velo_tavg'].append(test_pred_move_tavg)
        # lin_reg_results[align_event]['subspaces_tavg'].append(subspace)
        # lin_reg_results[align_event]['test_label_idx_tavg'].append(test_label_idx)

In [ ]:
# look at movement onset regressed against cursor position
neural_onset_idx_labels = ['delay_start_neural_idx', 'go_cue_neural_idx', 'mov_onset_neural_idx']
kin_onset_idx_labels = ['delay_start_kin_idx', 'go_cue_kin_idx', 'mov_onset_kin_idx']
nunits_to_use = np.min([len(stable_unit_idx[irec]) for irec in range(nrecs)])
for ievent, align_event in enumerate(align_events):
    lin_reg_results[align_event]['scores_pos_nmatch'] = []
    # lin_reg_results[align_event]['test_pos_nmatch'] = []
    # lin_reg_results[align_event]['test_pred_pos_nmatch'] = []
    for idate, date in enumerate(tqdm(dates)):
        neural_data = get_spike_seg_array(df[df['date']==date], neural_onset_idx_labels[ievent], samplerate=preproc_metadata['neural_samplerate'], smooth=True)
        kin_data = [cursor_seg[int(list(df[kin_onset_idx_labels[ievent]][df['date']==date])[iseg]):] for iseg, cursor_seg in enumerate(df['cursor_traj'][df['date']==date]) if np.array(df['good_trial'][df['date']==date])[iseg]]
        
        # Down sample kinematic data to match neural data
        if preproc_metadata['kin_samplerate'] != preproc_metadata['neural_samplerate']:
            dwns_kin_data = [aopy.precondition.base.downsample(kin_data[iseg], preproc_metadata['kin_samplerate'], preproc_metadata['neural_samplerate']) for iseg in range(len(kin_data))]
        
        temp_lin_reg_scores = np.zeros((nfolds, 2, niter_match))*np.nan
        for ii in range(niter_match):
            unit_idx_temp = np.random.choice(stable_unit_idx[idate], size=nunits_to_use, replace=False)
        
            stable_neural_data = [neural_data[itrial][:,unit_idx_temp] for itrial in range(len(neural_data)) if np.array(df['good_trial'][df['date']==date])[itrial]]
            scores, _, _, _ = xval_task_rel_dims(stable_neural_data, dwns_kin_data,regularization='ridge', nfolds=nfolds)
            temp_lin_reg_scores[:,:,ii] = scores
            
        lin_reg_results[align_event]['scores_pos_nmatch'].append(temp_lin_reg_scores)
        # lin_reg_results[align_event]['test_pos_nmatch'].append(test_move)
        # lin_reg_results[align_event]['test_pred_pos_nmatch'].append(test_pred_move)

In [ ]:
# look at movement onset regressed against cursor velocity
neural_onset_idx_labels = ['delay_start_neural_idx', 'go_cue_neural_idx', 'mov_onset_neural_idx']
kin_onset_idx_labels = ['delay_start_kin_idx', 'go_cue_kin_idx', 'mov_onset_kin_idx']
nunits_to_use = np.min([len(stable_unit_idx[irec]) for irec in range(nrecs)])
for ievent, align_event in enumerate(align_events):
    lin_reg_results[align_event]['scores_velo_nmatch'] = []
    # lin_reg_results[align_event]['test_velo_nmatch'] = []
    # lin_reg_results[align_event]['test_pred_velo_nmatch'] = []
    for idate, date in enumerate(tqdm(dates)):
        neural_data = get_spike_seg_array(df[df['date']==date], neural_onset_idx_labels[ievent], samplerate=100, smooth=True)
        
        kin_data = [cursor_seg[int(list(df[kin_onset_idx_labels[ievent]][df['date']==date])[iseg]):] for iseg, cursor_seg in enumerate(df['cursor_vel_traj'][df['date']==date]) if np.array(df['good_trial'][df['date']==date])[iseg]]
        # Down sample kinematic data to match neural data
        if preproc_metadata['kin_samplerate'] != preproc_metadata['neural_samplerate']:
            dwns_kin_data = [aopy.precondition.base.downsample(kin_data[iseg], preproc_metadata['kin_samplerate'], preproc_metadata['neural_samplerate']) for iseg in range(len(kin_data))]
        
        temp_lin_reg_scores = np.zeros((nfolds, 2, niter_match))*np.nan
        for ii in range(niter_match):
            unit_idx_temp = np.random.choice(stable_unit_idx[idate], size=nunits_to_use, replace=False)
        
            stable_neural_data = [neural_data[itrial][:,unit_idx_temp] for itrial in range(len(neural_data)) if np.array(df['good_trial'][df['date']==date])[itrial]]
            scores, _, _, _ = xval_task_rel_dims(stable_neural_data, dwns_kin_data,regularization='ridge', nfolds=nfolds)
            temp_lin_reg_scores[:,:,ii] = scores
            
        lin_reg_results[align_event]['scores_velo_nmatch'].append(temp_lin_reg_scores)
        # lin_reg_results[align_event]['test_pos_nmatch'].append(test_move)
        # lin_reg_results[align_event]['test_pred_pos_nmatch'].append(test_pred_move)

# Calculate statistics

## LDA 

In [ ]:
# define ecog decoding accuracy as one variable to make it easier to do further analysis
pp_column_sites = [column_units[igroup]['rec_site'][0] for igroup in range(len(column_units))]
pp_column_implants = [column_units[igroup]['implant'][0] for igroup in range(len(column_units))]
ecog_dec_acc_rec_site, ecog_dec_acc_rec_site_pp = {}, {}
ecog_dec_acc_rec_site_x = {}
ecog_dec_acc_rec_site_y = {}
for align_event in align_events:
    ecog_dec_acc_rec_site[align_event] = [ecog_dec_acc[subject][f"{implants[irec]}_interp"][rec_site-1] for irec, rec_site in enumerate(recording_site)]
    ecog_dec_acc_rec_site_pp[align_event] = [ecog_dec_acc[subject][f"{pp_column_implants[irec]}_interp"][rec_site-1] for irec, rec_site in enumerate(pp_column_sites)]
    ecog_dec_acc_rec_site_x[align_event] = [ecog_dec_acc_x[subject][f"{implants[irec]}_interp"][rec_site-1] for irec, rec_site in enumerate(recording_site)]
    ecog_dec_acc_rec_site_y[align_event] = [ecog_dec_acc_y[subject][f"{implants[irec]}_interp"][rec_site-1] for irec, rec_site in enumerate(recording_site)]

In [ ]:
# Calculate max decoding accuracy for each alignment
lda_max_calc_window = (-0.25, 0.25)
lda_max_calc_startidx = np.where(preproc_metadata['trial_time_axis'] > lda_max_calc_window[0])[0][0]
lda_max_calc_stopidx = np.where(preproc_metadata['trial_time_axis'] > lda_max_calc_window[1])[0][0]
for align_event in align_events:
    lda_results[align_event]['neural_space_max'] = [np.max(np.mean(lda_results[align_event]['neural_space'][irec][lda_max_calc_startidx:lda_max_calc_stopidx,:], axis=1)) for irec in range(nrecs)]

In [ ]:
# Fit linear regression to max
M1_rec_idx = np.where(np.in1d(recording_site, recording_brain_areas['M1']))[0]
PM_rec_idx = np.where(np.in1d(recording_site, recording_brain_areas['PM']))[0]
for align_event in align_events:
    _, _, lda_results[align_event]['neural_max_pcc'], lda_results[align_event]['neural_max_pcc_pval'], lda_results[align_event]['neural_max_reg_fit'] = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event]), 100*np.array(lda_results[align_event]['neural_space_max']))
    _, _, lda_results[align_event]['neural_max_pcc_M1'], lda_results[align_event]['neural_max_pcc_pval_M1'], lda_results[align_event]['neural_max_reg_fit_M1'] = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx], 100*np.array(lda_results[align_event]['neural_space_max'])[M1_rec_idx])
    _, _, lda_results[align_event]['neural_max_pcc_PM'], lda_results[align_event]['neural_max_pcc_pval_PM'], lda_results[align_event]['neural_max_reg_fit_PM'] = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx], 100*np.array(lda_results[align_event]['neural_space_max'])[PM_rec_idx])


In [ ]:
# # Trial adding
# # Calculate max decoding accuracy for each alignment
# for align_event in align_events:
#     lda_results_tradd[align_event]['neural_space_max'] = np.zeros((nrecs, ntrial_adding_bins))*np.nan
#     for itradd_bin in range(ntrial_adding_bins):
#         lda_results_tradd[align_event]['neural_space_max'][:,itradd_bin] = [np.max(np.mean(lda_results_tradd[align_event]['neural_space'][itradd_bin][irec], axis=1)) for irec in range(nrecs)]

In [ ]:
# # Trial adding
# for align_event in align_events:
#     lda_results_tradd[align_event]['neural_max_pcc'] = []
#     lda_results_tradd[align_event]['neural_max_pcc_pval'] = []
#     lda_results_tradd[align_event]['neural_max_reg_fit'] = []
#     lda_results_tradd[align_event]['neural_max_pcc_M1'] = []
#     lda_results_tradd[align_event]['neural_max_pcc_pval_M1'] = []
#     lda_results_tradd[align_event]['neural_max_reg_fit_M1'] = []
#     lda_results_tradd[align_event]['neural_max_pcc_PM'] = []
#     lda_results_tradd[align_event]['neural_max_pcc_pval_PM'] = []
#     lda_results_tradd[align_event]['neural_max_reg_fit_PM'] = []
#     for itradd_bin in range(ntrial_adding_bins):
#         _, _, temp1, temp2, temp3 = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event]), 100*np.array(lda_results_tradd[align_event]['neural_space_max'][:,itradd_bin]))
#         _, _, temp1_M1, temp2_M1, temp3_M1 = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx], 100*np.array(lda_results_tradd[align_event]['neural_space_max'][:,itradd_bin])[M1_rec_idx])
#         _, _, temp1_PM, temp2_PM, temp3_PM = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx], 100*np.array(lda_results_tradd[align_event]['neural_space_max'][:,itradd_bin])[PM_rec_idx])
#         lda_results_tradd[align_event]['neural_max_pcc'].append(temp1)
#         lda_results_tradd[align_event]['neural_max_pcc_pval'].append(temp2)
#         lda_results_tradd[align_event]['neural_max_reg_fit'].append(temp3)
#         lda_results_tradd[align_event]['neural_max_pcc_M1'].append(temp1_M1)
#         lda_results_tradd[align_event]['neural_max_pcc_pval_M1'].append(temp2_M1)
#         lda_results_tradd[align_event]['neural_max_reg_fit_M1'].append(temp3_M1)
#         lda_results_tradd[align_event]['neural_max_pcc_PM'].append(temp1_PM)
#         lda_results_tradd[align_event]['neural_max_pcc_pval_PM'].append(temp2_PM)
#         lda_results_tradd[align_event]['neural_max_reg_fit_PM'].append(temp3_PM)

## Linear regression

### Trial Averaged

In [ ]:
for align_event in align_events:
    mae_temp = np.array([np.mean(lin_reg_results[align_event]['mae_tavg_nmatch'][irec]) for irec in range(nrecs)])
    _, _, lin_reg_results[align_event]['tavg_nmatch_pcc'], lin_reg_results[align_event]['tavg_nmatch_pcc_pval'], lin_reg_results[align_event]['tavg_nmatch_reg_fit'] = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event]), mae_temp)
    _, _, lin_reg_results[align_event]['tavg_nmatch_pcc_M1'], lin_reg_results[align_event]['tavg_nmatch_pcc_pval_M1'], lin_reg_results[align_event]['tavg_nmatch_reg_fit_M1'] = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx], mae_temp[M1_rec_idx])
    _, _, lin_reg_results[align_event]['tavg_nmatch_pcc_PM'], lin_reg_results[align_event]['tavg_nmatch_pcc_pval_PM'], lin_reg_results[align_event]['tavg_nmatch_reg_fit_PM'] = aopy.analysis.base.linear_fit_analysis2D(100*np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx], mae_temp[PM_rec_idx])

    mae_temp_dist = np.array([lin_reg_results[align_event]['mae_tavg_nmatch'][irec] for irec in range(nrecs)])
    _, _, lin_reg_results[align_event]['tavg_nmatchdist_pcc'], lin_reg_results[align_event]['tavg_nmatchdist_pcc_pval'], lin_reg_results[align_event]['tavg_nmatchdist_reg_fit'] = aopy.analysis.base.linear_fit_analysis2D(np.tile(100*np.array(ecog_dec_acc_rec_site[align_event]),(niter_match,1)).T.flatten(), mae_temp_dist.flatten())
    _, _, lin_reg_results[align_event]['tavg_nmatchdist_pcc_M1'], lin_reg_results[align_event]['tavg_nmatchdist_pcc_pval_M1'], lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_M1'] = aopy.analysis.base.linear_fit_analysis2D(np.tile(100*np.array(ecog_dec_acc_rec_site[align_event]),(niter_match,1)).T[M1_rec_idx,:].flatten(), mae_temp_dist[M1_rec_idx,:].flatten())
    _, _, lin_reg_results[align_event]['tavg_nmatchdist_pcc_PM'], lin_reg_results[align_event]['tavg_nmatchdist_pcc_pval_PM'], lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_PM'] = aopy.analysis.base.linear_fit_analysis2D(np.tile(100*np.array(ecog_dec_acc_rec_site[align_event]),(niter_match,1)).T[PM_rec_idx,:].flatten(), mae_temp_dist[PM_rec_idx,:].flatten())
    

# Plot results and save

## LDA

### nLag analysis

In [ ]:
fig, ax = plt.subplots(1,nrecs,figsize=(nrecs*2,3))
for irec in range(nrecs):
    grad = aopy.visualization.get_color_gradient_RGB(len(test_lags), [0,0,0], [0.5,0.5,0.5])
    [ax[irec].plot(np.mean(lda_results['lag_test']['scores'][f"site{irec}"][ilag], axis=1), color=grad[ilag], label=str(ilag)) for ilag in range(len(lda_results['lag_test']['scores'][f"site{irec}"]))]
    ax[irec].legend()
    ax[irec].set_title(f"Recording {recording_site[irec]}", color=day_colors[irec])
fig.tight_layout()
plt.show()

### All good units in all recordings

In [ ]:
# plot how many good units are in each and their average FR
print("Recording Site:", recording_site)
print("n Stable Units:", nstable_unit)
fig, ax = plt.subplots(1,3,figsize=(14,3))
ax[0].bar(np.arange(nrecs), nstable_unit)
ax[0].set(ylabel='Number of stable units', xlabel='Recording Site'), ax[0].set_xticks(np.arange(nrecs), recording_site)
nM1_units = [nstable_unit[irec] for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
nPM_units = [nstable_unit[irec] for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
ax[1].bar(np.arange(len(nM1_units)), nM1_units), ax[2].bar(np.arange(len(nPM_units)), nPM_units)
ax[1].set(ylabel='Number of stable units',xlabel='Recording Site', title='M1'), ax[1].set_xticks(np.arange(len(nM1_units)), recording_site[np.in1d(recording_site, recording_brain_areas['M1'])])
ax[2].set(ylabel='Number of stable units',xlabel='Recording Site', title='PM'), ax[2].set_xticks(np.arange(len(nPM_units)), recording_site[np.in1d(recording_site, recording_brain_areas['PM'])])
fig.tight_layout()
plt.show()

In [ ]:
# Plot average FR on units
fr_colors = [[1,1,1], [0.75, 0.75, 0.75], [0.5, 0.5, 0.5]]
fig, ax = plt.subplots(1,1, figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    test = []
    test = [np.mean(rasters['neural'][align_event][irec][:,np.array(df['good_trial'][df['date']==dates[irec]]),:][:,:,stable_unit_idx[irec]], axis=(0,1)) for irec in range(nrecs)]
    # aopy.visualization.plot_boxplots(test, list(np.arange(nrecs)+(0.25*ievent)), trendline=False, ax=ax)
    ax.set_xticks([], [])
    aopy.visualization.plot_boxplots(test, np.arange(len(test))+0.25*ievent, trendline=False, facecolor=fr_colors[ievent], box_width=0.2, ax=ax)
ax.set_xticks(np.arange(len(test))+0.25, recording_site), ax.set(xlabel='Recording Site', ylabel='FR')

fig.tight_layout()
plt.show()

fig, ax = plt.subplots(1,2, figsize=(12,3))
for ievent, align_event in enumerate(align_events):
    test_M1 = [np.mean(rasters['neural'][align_event][irec][:,np.array(df['good_trial'][df['date']==dates[irec]]),:][:,:,stable_unit_idx[irec]], axis=(0,1)) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    test_PM = [np.mean(rasters['neural'][align_event][irec][:,np.array(df['good_trial'][df['date']==dates[irec]]),:][:,:,stable_unit_idx[irec]], axis=(0,1)) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    ax[0].set_xticks([], []), ax[1].set_xticks([], [])
    aopy.visualization.plot_boxplots(test_M1, np.arange(len(test_M1))+0.25*ievent, trendline=False, facecolor=fr_colors[ievent], box_width=0.2, ax=ax[0])
    aopy.visualization.plot_boxplots(test_PM, np.arange(len(test_PM))+0.25*ievent, trendline=False, facecolor=fr_colors[ievent], box_width=0.2, ax=ax[1])


ax[0].set_xticks(np.arange(len(test_M1))+0.25, recording_site[np.in1d(recording_site, recording_brain_areas['M1'])]), ax[0].set(xlabel='Recording Site', ylabel='FR', title='M1')
ax[1].set_xticks(np.arange(len(test_PM))+0.25, recording_site[np.in1d(recording_site, recording_brain_areas['PM'])]), ax[1].set(xlabel='Recording Site', ylabel='FR', title='PM')

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    max_planning_dec_acc =  np.max(np.mean(lda_results[align_event]['all_units_score'], axis=1)[trelevant1:trelevant2])
    ax[ievent].plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['all_units_score'], axis=1))
    ax[ievent].set(xlabel='Time [s]', ylabel='Decoding Accuracy [%]', title=f"{align_event}: Max: {np.round(np.max(max_planning_dec_acc),3)}", ylim=(0.125,1)) 
    ax[ievent].plot([trelevant1_time,trelevant1_time], [0,1], 'r--')
    ax[ievent].plot([trelevant2_time,trelevant2_time], [0,1], 'r--')
fig.tight_layout()
plt.show()

In [ ]:
nimportant_units = 20
nall_units = lda_results[align_event]['all_units_weight'].shape[2]
rel_neuron_depth = np.concatenate([3840-neuron_pos[irec] - np.min(3840-neuron_pos[irec]) for irec in range(nrecs)])
all_sorted_weights = []
for align_event in align_events:
    fig, ax = plt.subplots(1,3,figsize=(12,3.5))
    
    sorted_weights = np.argsort(np.sum(np.max(np.abs(np.mean(lda_results[align_event]['all_units_weight'], axis=3))[trelevant1:trelevant2,:,:], axis=1), axis=0))
    all_sorted_weights.append(sorted_weights)
    ax0 = ax[0].pcolor(preproc_metadata['trial_time_axis'], np.arange(nall_units), np.max(np.abs(np.mean(lda_results[align_event]['all_units_weight'], axis=3)), axis=1)[:,sorted_weights].T, vmin=0, vmax=75)
    ax[0].plot([trelevant1_time,trelevant1_time], [0,nall_units], 'r--')
    ax[0].plot([trelevant2_time,trelevant2_time], [0,nall_units], 'r--')
    ax[0].set(title='Ordered by total weight', ylim=(0,nall_units), xlabel='Time [s]')
    cb = plt.colorbar(ax0)

    ax[1].plot(np.arange(nall_units), np.sum(np.max(np.abs(np.mean(lda_results[align_event]['all_units_weight'], axis=3))[trelevant1:trelevant2,:,sorted_weights], axis=1), axis=0))
    ax[1].set(title=f'{trelevant1_time}-{trelevant2_time}s after event', xlabel='Unit', ylabel='Weight')
    ax[1].plot([nall_units-nimportant_units, nall_units-nimportant_units], ax[1].get_ylim(), 'r--')
    
    ax1 = ax[2].pcolor(preproc_metadata['trial_time_axis'], np.arange(nall_units), np.max(np.abs(np.mean(lda_results[align_event]['all_units_weight'], axis=3)), axis=1)[:,np.argsort(rel_neuron_depth)].T, vmin=0, vmax=60)
    ax[2].set_yticks([0, nall_units], [np.min(rel_neuron_depth), np.max(rel_neuron_depth)])
    ax[2].plot([trelevant1_time,trelevant1_time], [0,nall_units], 'r--')
    ax[2].plot([trelevant2_time,trelevant2_time], [0,nall_units], 'r--')
    ax[2].set(title='Ordered by depth', ylim=(0,nall_units), xlabel='Time [s]', ylabel='Depth (nonlinear) [um]')
    cb = plt.colorbar(ax1)
    
    plt.suptitle(align_event)
    fig.tight_layout()
    plt.show()
    

In [ ]:
all_unit_recoring_number = np.concatenate([irec*np.ones(len(stable_unit_idx[irec])) for irec in range(nrecs)])
fig, ax = plt.subplots(1,3, figsize=(14, 2.5))
overlap_unit_idx = all_sorted_weights[0][-nimportant_units:][np.in1d(all_sorted_weights[0][-nimportant_units:], all_sorted_weights[2][-nimportant_units:])]
for ievent, event in enumerate(align_events):
    important_units = all_sorted_weights[ievent][-nimportant_units:]
    units_per_rec = []
    for irec in range(len(np.unique(all_unit_recoring_number))):
        units_per_rec.append(np.sum(all_unit_recoring_number[important_units] == irec))
        
        
    ax[ievent].bar(np.arange(len(units_per_rec)), units_per_rec)
    ax[ievent].set(xlabel='Recording Site', ylabel='Number of units', title=f"{event} - Top {nimportant_units} units", ylim=(0,15))
    ax[ievent].set_xticks(np.arange(len(units_per_rec)), recording_site)
fig.tight_layout()
plt.show()

In [ ]:
all_unit_recoring_number = np.concatenate([irec*np.ones(len(stable_unit_idx[irec])) for irec in range(nrecs)])
fig, ax = plt.subplots(1,3, figsize=(10, 4))
overlap_unit_idx = all_sorted_weights[0][-nimportant_units:][np.in1d(all_sorted_weights[0][-nimportant_units:], all_sorted_weights[2][-nimportant_units:])]
for ievent, event in enumerate(align_events):
    important_units = all_sorted_weights[ievent][-nimportant_units:]
    
    for unit_idx in important_units:
        if unit_idx in overlap_unit_idx:
            ax[ievent].plot(np.sum(np.max(np.abs(np.mean(lda_results[align_event]['all_units_weight'], axis=3)), axis=1)[trelevant1:trelevant2,unit_idx], axis=0), rel_neuron_depth[unit_idx],  '*', color=day_colors[int(all_unit_recoring_number[unit_idx])])
        else:
            ax[ievent].plot(np.sum(np.max(np.abs(np.mean(lda_results[align_event]['all_units_weight'], axis=3)), axis=1)[trelevant1:trelevant2,unit_idx], axis=0), rel_neuron_depth[unit_idx],  '.', color=day_colors[int(all_unit_recoring_number[unit_idx])])
    ax[ievent].invert_yaxis()
    ax[ievent].set(xlabel='Weight', ylabel='Depth [um]', title=f"{event} - Top {nimportant_units} units", xlim=(100,1050))
fig.tight_layout()
plt.show()

#### Single channel decoding

In [ ]:
align_event = align_events[-1]
fig, ax = plt.subplots(1,nrecs, figsize=(nrecs*3,4))
for irec in range(nrecs):
    try:
        cbdsd = ax[irec].pcolor(preproc_metadata['trial_time_axis'], np.arange(len(stable_unit_idx[irec])), np.mean(lda_results[align_event]['single_ch_decoding'][irec],axis=2).T, vmin=0.125, vmax=0.4)
        cb = plt.colorbar(cbdsd)
        ax[irec].set(title=f"Site: {recording_site[irec]}", xlabel='Time [s]', ylabel = 'Unit Number')
    except:
        continue

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10,4))
nch = lda_results[align_event]['single_ch_decoding'][0].shape[1]
for irec in range(nrecs):
    try:
        nch = np.mean(lda_results[align_event]['single_ch_decoding'][irec],axis=2).shape[1]
        maxtidx = np.argmax(np.mean(lda_results[align_event]['single_ch_decoding'][irec], axis=2), axis=0)
        ax.plot(np.flip(np.sort([np.mean(lda_results[align_event]['single_ch_decoding'][irec],axis=2)[maxtidx[ich],ich] for ich in range(nch)])), color=day_colors[irec], linewidth=4)
        ax.set(xlabel='Sorted Channel', ylabel='Decoding Accuracy')
    except:
        continue
fig.tight_layout()
plt.show()

In [ ]:
align_event = align_events[-1]
single_ch_max_dec = []
single_ch_site = []
for ii in range(len(lda_results[align_event]['single_ch_decoding'])):
    if len(lda_results[align_event]['single_ch_decoding'][ii]) > 0:
        for iunit in range(lda_results[align_event]['single_ch_decoding'][ii].shape[1]):
            maxidx = np.argmax(np.mean(lda_results[align_event]['single_ch_decoding'][ii][:,iunit,:], axis=1))
            single_ch_max_dec.append(100*np.mean(lda_results[align_event]['single_ch_decoding'][ii][:,iunit,:], axis=1)[maxidx])
            single_ch_site.append(ii)
    

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,5), gridspec_kw={'width_ratios':[1,3]})
single_ch_med = np.median(single_ch_max_dec)
single_ch_sd = np.std(single_ch_max_dec)
high_dec_mask = np.array(single_ch_max_dec) > single_ch_med+1*single_ch_sd
hist, bins = np.histogram(np.array(single_ch_site)[high_dec_mask], bins = np.arange(nrecs+1)-.5)
ax[0].hist(single_ch_max_dec, bins=np.arange(8,50,1), color=(.498, 0.098, .184), alpha=0.5)
ax[0].hist(np.array(single_ch_max_dec)[high_dec_mask], bins=np.arange(8,50,1), color=(.498, 0.098, .184))
# ax[0].plot([single_ch_med, single_ch_med], ax[0].get_ylim())
ax[0].plot([single_ch_med+1*single_ch_sd, single_ch_med+1*single_ch_sd], ax[0].get_ylim(), 'r', linewidth=5)
ax[0].set(xlabel='Single Unit \n Accuracy [%]', ylabel='Unit Count')
ax[0].set_xticks([15,30,45])
colors_hist = np.array(hist)
dec_map = ax[1].scatter(ecog_dec_acc[subject]['rec_locations'][:,0], ecog_dec_acc[subject]['rec_locations'][:,1], s=250, c=colors_hist/hist_denom, cmap='YlGnBu')
cb = plt.colorbar(dec_map)
# cb.set_ticks((0,.5))
# cb.set_label('High decoding units', color=(.498, 0.098, .184))
ax[1].set(xlim=(-6,6), ylim=(-6,6), xlabel='X [mm]', ylabel='Y [mm]')
ax[1].set_aspect('equal')

fig.tight_layout()
aopy.visualization.savefig(base_save_dir, 'surface_distribution.svg')

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(4,6.25))
nch = lda_results[align_event]['single_ch_decoding'][0].shape[1]
dec_thresh = single_ch_med+1*single_ch_sd
for irec in range(nrecs):
    nch = np.mean(lda_results[align_event]['single_ch_decoding'][irec],axis=2).shape[1]
    maxtidx = np.argmax(np.mean(lda_results[align_event]['single_ch_decoding'][irec], axis=2), axis=0)
    rel_depth_list = list(unit_df[unit_df['rec_number']==irec]['rel_depth'])
    temp = []
    for ich in range(nch):
        temp.append(100*np.mean(lda_results[align_event]['single_ch_decoding'][irec],axis=2)[maxtidx[ich],ich])
    ax.plot(np.array(temp)[np.argsort(rel_depth_list)],np.sort(rel_depth_list)/1000 ,'.', color=(.498, 0.098, .184), alpha=0.5, markersize=10)
    ax.plot(np.array(temp)[np.argsort(rel_depth_list)][np.array(temp)[np.argsort(rel_depth_list)] > dec_thresh], np.sort(rel_depth_list)[np.array(temp)[np.argsort(rel_depth_list)] > dec_thresh]/1000,'.', color=(.498, 0.098, .184), markersize=10)
    ax.set(ylabel='Depth [mm]', xlabel='Single Unit \n Accuracy [%]')
    ax.invert_yaxis()
fig.tight_layout()
aopy.visualization.savefig(base_save_dir, 'lda_single_ch_depth.svg')

### All good units

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events):
    max_planning_dec_acc_recs = [np.max(np.mean(lda_results[align_event]['scores'][irec], axis=1)[trelevant1:trelevant2]) for irec in range(nrecs)]
    [ax[ievent].plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['scores'][irec], axis=1), color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel='Time [s]', ylabel='Neuropixel Dec Acc', title=f"{align_event}: Max: {np.round(np.max(max_planning_dec_acc_recs),3)}",ylim=(0.125, 1))
    ax[ievent].legend(recording_site, bbox_to_anchor=(1,1), fontsize=6)
    ax[ievent].plot([0,0], [0,1], 'r--')
    ax[ievent].plot([0.2,0.2], [0,1], 'r--')
    # ax[ievent].annotate(f"Max: {np.round(np.max(max_planning_dec_acc_recs),3)}", (0.25, 0.4))
fig.tight_layout()
plt.show()

In [ ]:
align_event = align_events[-1]
for irec in range(nrecs):
    fig, ax = plt.subplots(1,2,figsize=(10,2))
    ax[0].plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['scores'][irec], axis=1))

    max_tidx = np.argmax(np.mean(lda_results[align_event]['scores'][irec], axis=1))
    ax[0].plot([preproc_metadata['trial_time_axis'][nlda_lags:][max_tidx], preproc_metadata['trial_time_axis'][nlda_lags:][max_tidx]], [0.1, 0.8], 'r--')
    # Look at weights at max t
    weightim = ax[1].pcolor(np.mean(lda_results[align_event]['weights'][irec], axis=3)[max_tidx,:,:])
    ax[0].set(xlabel='Time', ylabel='Dec Acc')
    ax[1].set(xlabel='Unit', ylabel='Target')
    plt.suptitle(f"Recording {recording_site[irec]}")
    cb = plt.colorbar(weightim, label='Weight')
    
    unit_rank = np.flip(np.argsort((np.sum(np.abs(np.mean(lda_results[align_event]['weights'][irec], axis=3)[max_tidx,:,:]), axis=0))))
    fig, ax = plt.subplots(1,1,figsize=(10,1.5))
    # weightim = ax[0].pcolor(np.mean(lda_results['weights'][irec], axis=3)[max_tidx,:,unit_rank].T, vmin=-10, vmax=10)
    # ax[0].set(xlabel='Unit number', ylabel='Target')
    # cb = plt.colorbar(weightim, label='Weight')
    ax.plot((np.sum(np.abs(np.mean(lda_results[align_event]['weights'][irec], axis=3)[max_tidx,:,:]), axis=0))[unit_rank])
    ax.set(xlabel='Unit number', ylabel='Total weight', xlim=(0,50))
    ax.set_xticks(np.arange(len(stable_unit_idx[irec]))[:50], stable_unit_labels[irec][unit_rank][:50], rotation = 75)
    fig.tight_layout()
    plt.show()
    
    # Plot FR vs. time traces of the 10 highest weighted neurons
    avg_fr_each_trial = np.mean(rasters['neural'][align_event][irec][:,:,stable_unit_idx[irec]], axis=0)
    smooth_avg_fr_each_trial = aopy.analysis.base.calc_rolling_average(avg_fr_each_trial, window_size=17)
    
    nunits_plot = 6
    fig, ax = plt.subplots(2,nunits_plot, figsize=(nunits_plot*1.6,3))
    for iunit in range(nunits_plot):
        ax[0,iunit].plot(smooth_avg_fr_each_trial[:ntest_trials,unit_rank[iunit]])
        ax[0,iunit].set(xlabel='Trial', ylabel='Firing rate', title=f"Unit: {stable_unit_labels[irec][unit_rank[iunit]]}")
        ax[1,iunit].plot(unit_df[unit_df['rec_number']==irec].reset_index()['waveform'][unit_rank[iunit]], 'r--')

    plt.suptitle(f"Recording {recording_site[irec]} - High weight units")
    fig.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(2,nunits_plot, figsize=(nunits_plot*1.6,3))
    for iunit in range(nunits_plot):
        ax[0,iunit].plot(smooth_avg_fr_each_trial[:ntest_trials,unit_rank[-(iunit+1)]])
        ax[1,iunit].set(xlabel='Trial', ylabel='Firing rate', title=f"Unit: {stable_unit_labels[irec][unit_rank[-(iunit+1)]]}")
        ax[1,iunit].plot(unit_df[unit_df['rec_number']==irec].reset_index()['waveform'][unit_rank[-(iunit+1)]], 'r--')

    plt.suptitle(f"Recording {recording_site[irec]} - Low weight units")
    fig.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(14,4))
[ax[0].plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['scores'][irec],axis=1), linewidth=4,color=day_colors[irec], label=f"{np.round(ecog_dec_acc_rec_site[align_event][irec],3)}") for irec in range(nrecs)]
[ax[1].plot(lda_results[align_event]['dec_unit_weight_rank'][irec], linewidth=4,color=day_colors[irec], label=f"{np.round(ecog_dec_acc_rec_site[align_event][irec],3)}") for irec in range(nrecs)]
ax[1].set(xlabel='Unit', ylabel='Total LDA weight')
ax[1].set_yscale('log')
ax[1].legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
nunitsplt = 30
fig, ax = plt.subplots(2,2,figsize=(14,6))
[ax[0,0].plot(lda_results[align_event]['dec_unit_weight_rank'][irec][:nunitsplt], linewidth=4,color=day_colors[irec]) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
[ax[0,1].plot(lda_results[align_event]['dec_unit_weight_rank'][irec][:nunitsplt], linewidth=4,color=day_colors[irec]) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]

[ax[1,0].plot(lda_results[align_event]['dec_unit_weight_rank'][irec][:nunitsplt]/np.max(lda_results[align_event]['dec_unit_weight_rank'][irec][:nunitsplt]), linewidth=4,color=day_colors[irec], label=f"{np.round(ecog_dec_acc_rec_site[align_event][irec],3)}") for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
[ax[1,1].plot(lda_results[align_event]['dec_unit_weight_rank'][irec][:nunitsplt]/np.max(lda_results[align_event]['dec_unit_weight_rank'][irec][:nunitsplt]), linewidth=4,color=day_colors[irec], label=f"{np.round(ecog_dec_acc_rec_site[align_event][irec],3)}") for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]

[ax[0, iax].set(xlabel='Unit', ylabel='LDA weight') for iax in range(ax.shape[0])]
[ax[1, iax].set(xlabel='Unit', ylabel='Normalized LDA weight') for iax in range(ax.shape[0])]
_, _ = ax[0,0].set(title='M1'), ax[0,1].set(title='PM')

[ax[1,iax].legend() for iax in range(ax.shape[0])]
plt.show()

In [ ]:
for ievent, align_event in enumerate(align_events):
    fig, ax = plt.subplots(1,nrecs, figsize=(20,4))
    for irec in range(nrecs):
        ax[irec].plot(np.mean(np.abs(lda_results[align_event]['weights'][irec]), axis=(0,1,3)), 3840-neuron_pos[irec], '.')
        ax[irec].set(xlabel='Weight', ylabel='', title=f'Rec Site: {recording_site[irec]}', xlim=(0, 12.5))
        ax[irec].invert_yaxis()
    plt.suptitle(f"{align_event}")
    fig.tight_layout()
    plt.show()

In [ ]:
for ievent, align_event in enumerate(align_events):
    fig, ax = plt.subplots(1,nrecs, figsize=(20,4))
    for irec in range(nrecs):
        ax[irec].plot(np.mean(np.abs(lda_results[align_event]['weights'][irec]), axis=(0,1,3)), 3840-neuron_pos[irec], '.')
        ax[irec].set(xlabel='Weight', ylabel='', title=f'Rec Site: {recording_site[irec]}', xlim=(0, 12.5))
        ax[irec].invert_yaxis()
    plt.suptitle(f"{align_event}")
    fig.tight_layout()
    plt.show()

In [ ]:
for ievent, align_event in enumerate(align_events):
    fig, ax = plt.subplots(1,nrecs, figsize=(20,4))
    for irec in range(nrecs):
        for itarget in range(len(np.unique(df['target_idx']))):
            ax[irec].plot(np.mean(np.abs(lda_results[align_event]['weights'][irec][:,itarget,:,:]), axis=(0,2)), 3840-neuron_pos[irec], '.',color=colors[itarget+1], alpha=0.5)
        ax[irec].set(xlabel='Weight', ylabel='', title=f'Rec Site: {recording_site[irec]}')
        ax[irec].invert_yaxis()
    plt.suptitle(f"{align_event}")
    fig.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots(1,nrecs, figsize=(40,3))
for irec in range(nrecs):
    max_decoding_time = np.argmax(np.mean(lda_results[align_event]['scores'][irec], axis=1))
    cm = ax[irec].pcolor(np.mean(lda_results[align_event]['confusion_matrix'][irec][max_decoding_time,:,:,:], axis=0), vmin=0.125, vmax=.7)
    cb = plt.colorbar(cm)
    ax[irec].set(title=f"Rec Site: {recording_site[irec]}")

fig.tight_layout()
plt.show()

fig, ax = plt.subplots(1,nrecs, figsize=(40,3))
for irec in range(nrecs):
    max_decoding_time = np.argmax(np.mean(lda_results[align_event]['scores'][irec], axis=1))
    cm = ax[irec].pcolor(np.mean(lda_results[align_event]['confusion_matrix'][irec][max_decoding_time,:,:,:], axis=0), vmin=0.125)
    cb = plt.colorbar(cm)
    ax[irec].set(title=f"Rec Site: {recording_site[irec]}")

fig.tight_layout()
plt.show()

### Neuron number matched

In [ ]:
# Plot all recording sites overlaid
for align_event in align_events:
    fig, ax = plt.subplots(1,3,figsize=(10, 3))
    [ax[0].plot(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], label=f'{np.round(ecog_dec_acc_rec_site[align_event][irec],3)}') for irec in range(nrecs)]
    [ax[0].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1),100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1)-aopy.analysis.calc_sem(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], alpha=0.5) for irec in range(nrecs)]
    [ax[0].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1),100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1)+aopy.analysis.calc_sem(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], alpha=0.5) for irec in range(nrecs)]
    
    [ax[1].plot(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], label=f'{np.round(ecog_dec_acc_rec_site[align_event][irec],3)}') for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[1].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1),100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1)-aopy.analysis.calc_sem(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], alpha=0.5) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[1].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1),100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1)+aopy.analysis.calc_sem(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], alpha=0.5) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]

    [ax[2].plot(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], label=f'{np.round(ecog_dec_acc_rec_site[align_event][irec],3)}') for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    [ax[2].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1),100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1)-aopy.analysis.calc_sem(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], alpha=0.5) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    [ax[2].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1),100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1)+aopy.analysis.calc_sem(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec], alpha=0.5) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]

    ax[0].set(xlabel='Time [s]', ylabel='Accuracy', ylim=(0,100), title='SEM error')
    ax[1].set(xlabel='Time [s]', ylabel='Accuracy', ylim=(0,100), title='M1')
    ax[2].set(xlabel='Time [s]', ylabel='Accuracy', ylim=(0,100), title='PM')
    # [ax[ia].legend(loc = 'upper left', fontsize=6) for ia in range(len(ax))]
    [ax[ia].spines[['right','top']].set_visible(False) for ia in range(len(ax))]
    fig.suptitle(f"{align_event}")
    fig.tight_layout()
    # if save_figs:
    #     aopy.visualization.savefig(save_dir, 'lda_decoding_matched_pop_sem_error.svg')

In [ ]:
if subject == 'beignet':
    align_event=align_events[1]
    fig, ax = plt.subplots(1,1,figsize=(3,2.5))
    [ax.plot(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec]) for irec in np.where(recording_site==11)[0]]
    [ax.plot(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec]) for irec in [np.where(recording_site==9)[0][1]]]
    # [ax.plot(preproc_metadata['trial_time_axis'][nlda_lags:], 100*np.mean(lda_results[align_event]['neural_space'][irec], axis=1), color=day_colors[irec]) for irec in np.where(recording_site==30)[0]]
    ax.set(xlabel='Time [s]', ylabel='Accuracy [%]', ylim=(12.5,50))

    fig.tight_layout()
    aopy.visualization.savefig(base_save_dir, f'lda_repeated_penetrations.svg')

In [ ]:
# Plot max decoding
for align_event in align_events:
    fig, ax = plt.subplots(1,3,figsize=(10,3))
    [ax[0].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[iday], 100*np.array(lda_results[align_event]['neural_space_max'])[iday], '.', color=day_colors[iday], markersize=12) for iday in range(nrecs)]
    ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (2e7, 60))
    ax[0].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [%]', ylim=(10, 75))
    
    [ax[1].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[iday], 100*np.array(lda_results[align_event]['neural_space_max'])[iday], '.', color=day_colors[iday], markersize=12) for iday in range(nrecs) if recording_site[iday] in recording_brain_areas['M1']]
    ax[1].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])], [(lda_results[align_event]['neural_max_reg_fit_M1'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lda_results[align_event]['neural_max_reg_fit_M1'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit_M1'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lda_results[align_event]['neural_max_reg_fit_M1'].intercept_)[0]])               
    ax[1].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval_M1'],3)}", (2e7, 60))
    ax[1].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [%]', ylim=(10, 75), title='M1')
    
    [ax[2].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[iday], 100*np.array(lda_results[align_event]['neural_space_max'])[iday], '.', color=day_colors[iday], markersize=12) for iday in range(nrecs) if recording_site[iday] in recording_brain_areas['PM']]
    ax[2].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])], [(lda_results[align_event]['neural_max_reg_fit_PM'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lda_results[align_event]['neural_max_reg_fit_PM'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit_PM'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lda_results[align_event]['neural_max_reg_fit_PM'].intercept_)[0]])               
    ax[2].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval_PM'],3)}", (2e7, 60))
    ax[2].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [%]', ylim=(10, 75), title='PM')
    
    fig.suptitle(f"{align_event}")
    fig.tight_layout()
    
        # if save_figs:
    aopy.visualization.savefig(base_save_dir, f'lda_dec_acc_{align_event}.svg')

In [ ]:
## Plot spatial map of decoding
fig, ax = plt.subplots(1,3,figsize=(14,4))
for ievent, align_event in enumerate(align_events):
    colors = np.array(lda_results[align_event]['neural_space_max'])
    dec_map = ax[ievent].scatter(ecog_dec_acc[subject]['rec_locations'][:,0], ecog_dec_acc[subject]['rec_locations'][:,1], s=100, c=colors, cmap='YlGnBu', vmin=0.2, vmax=.4)
    cb = plt.colorbar(dec_map)
    ax[ievent].set(xlim=(-6,6), ylim=(-6,6), title=align_event)
    
fig.tight_layout()
plt.show()

In [ ]:
# Plot max decoding
tline_xmax = 7e7
tline_xmin = 0
for align_event in align_events:
    fig, ax = plt.subplots(1,1,figsize=(3,3))
    m1_color='firebrick'
    pm_color='black'
    ax.plot(100*np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx], 100*np.array(lda_results[align_event]['neural_space_max'])[M1_rec_idx], '.', color=m1_color, markersize=10)
    ax.plot(100*np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx], 100*np.array(lda_results[align_event]['neural_space_max'])[PM_rec_idx], '.', color=pm_color, markersize=10)
    ax.plot([tline_xmin, tline_xmax], [(lda_results[align_event]['neural_max_reg_fit_M1'].coef_*tline_xmin+lda_results[align_event]['neural_max_reg_fit_M1'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit_M1'].coef_*tline_xmax+lda_results[align_event]['neural_max_reg_fit_M1'].intercept_)[0]], color=m1_color, linewidth=4)               
    ax.plot([tline_xmin, tline_xmax], [(lda_results[align_event]['neural_max_reg_fit_PM'].coef_*tline_xmin+lda_results[align_event]['neural_max_reg_fit_PM'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit_PM'].coef_*tline_xmax+lda_results[align_event]['neural_max_reg_fit_PM'].intercept_)[0]], color=pm_color, linewidth=4)               
    ax.annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (2e7, 60))
    ax.set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [%]', ylim=(0, 75), title=f"{align_event}")    
        # if save_figs:
    aopy.visualization.savefig(base_save_dir, f'lda_dec_acc_{align_event}.svg')

### KS drift

In [ ]:
# Plot max decoding
for align_event in align_events:
    fig, ax = plt.subplots(1,3,figsize=(10,3))
    ax[0].plot(np.array(ksdrift['drift_max']), 100*np.array(lda_results[align_event]['neural_space_max']), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[0].set(xlabel='KS Drift [um]', ylabel='Neuropixel Acc [%]', ylim=(10, 100))
    
    ax[1].plot(np.array(ksdrift['drift_max'])[M1_rec_idx], 100*np.array(lda_results[align_event]['neural_space_max'])[M1_rec_idx], 'k.', markersize=8)
    # ax[1].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])], [(lda_results[align_event]['neural_max_reg_fit_M1'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lda_results[align_event]['neural_max_reg_fit_M1'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit_M1'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lda_results[align_event]['neural_max_reg_fit_M1'].intercept_)[0]])               
    # ax[1].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval_M1'],3)}", (3, 20))
    ax[1].set(xlabel='KS Drift [um]', ylabel='Neuropixel Acc [%]', ylim=(10, 100), title='M1')
    
    ax[2].plot(np.array(ksdrift['drift_max'])[PM_rec_idx], 100*np.array(lda_results[align_event]['neural_space_max'])[PM_rec_idx], 'k.', markersize=8)
    # ax[2].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])], [(lda_results[align_event]['neural_max_reg_fit_PM'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lda_results[align_event]['neural_max_reg_fit_PM'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit_PM'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lda_results[align_event]['neural_max_reg_fit_PM'].intercept_)[0]])               
    # ax[2].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval_PM'],3)}", (3, 20))
    ax[2].set(xlabel='KS Drift [um]', ylabel='Neuropixel Acc [%]', ylim=(10, 100), title='PM')
    
    fig.suptitle(f"{align_event}")
    fig.tight_layout()

### Pseudopopulation

#### Random

In [ ]:
print(pseudopopulation_metadata['nrandom_units'])
for align_event in align_events:
    random_scores = [np.mean(lda_results[align_event]['pseudopopulations']['random_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['random_scores']))]
    fig, ax = plt.subplots(1,3,figsize=(13,3.5))
    ax[0].plot(preproc_metadata['trial_time_axis'][nlda_lags:], np.vstack(random_scores).T)
    [ax[iax].set(xlabel='Time [s]', ylabel='Decoding Accuracy', ylim=(0.125,1), title=align_event) for iax in range(len(ax))]
    random_mean = np.mean(random_scores, axis=0)
    random_std = np.std(random_scores, axis=0)
    ax[1].plot(preproc_metadata['trial_time_axis'][nlda_lags:], np.mean(random_scores, axis=0), color='k')
    ax[1].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], random_mean+random_std, random_mean, color='k', alpha=0.5)
    ax[1].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], random_mean-random_std, random_mean, color='k', alpha=0.5)
    ax[1].set(title=f"{align_event} - {pseudopopulation_metadata['nrandom_groups']} iteration $\mu \pm \sigma$")
    
    random_max = np.max(random_scores, axis=0)
    random_min = np.min(random_scores, axis=0)
    ax[2].plot(preproc_metadata['trial_time_axis'][nlda_lags:], np.mean(random_scores, axis=0), color='b')
    ax[2].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], random_max, random_mean, color='b', alpha=0.5)
    ax[2].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], random_min, random_mean, color='b', alpha=0.5)
    ax[2].set(title=f"{align_event} - {pseudopopulation_metadata['nrandom_groups']} iteration envelope")
    
    plt.suptitle(f"Random Unit Groups {align_event}")
    fig.tight_layout()
    plt.show()

#### Column

In [ ]:
column_colors = []
for irec in range(len(column_units)):
    if len(column_units[irec]) > 0:
        date_mask = np.logical_and(ecog_dec_acc[subject]['rec_locations'][:,0] == column_units[irec]['rec_xpos'][0], ecog_dec_acc[subject]['rec_locations'][:,1] == column_units[irec]['rec_ypos'][0])
        # print(column_units[irec]['rec_xpos'][0], column_units[irec]['rec_ypos'][0])
        # print(column_units[irec]['rec_rcaxis'][0], day_colors[date_idx])
        date_idx = np.where(date_mask)[0][0]
        # print(date_idx, day_colors[date_idx])
        column_colors.append(day_colors[date_idx])
        
    else:
        # column_colors.append([0,0,0])
        continue

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
column_scores = [np.mean(lda_results[align_event]['pseudopopulations']['column_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
column_scores_std = [np.std(lda_results[align_event]['pseudopopulations']['column_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
column_scores_max = [np.max(lda_results[align_event]['pseudopopulations']['column_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
column_scores_min = [np.min(lda_results[align_event]['pseudopopulations']['column_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
column_sites = [column_units[igroup]['rec_site'][0] for igroup in range(len(column_units)) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
fig, ax = plt.subplots(1,3,figsize=(13,3.5))

for igroup in range(len(column_sites)):
    ax[0].plot(preproc_metadata['trial_time_axis'], np.vstack(column_scores)[igroup,:], color=column_colors[igroup])

ax[0].legend(column_sites, bbox_to_anchor=(1,1))
for igroup in range(len(column_sites)):
    ax[0].fill_between(preproc_metadata['trial_time_axis'], column_scores[igroup]+column_scores_std[igroup],column_scores[igroup], alpha=0.5, color=column_colors[igroup])
    ax[0].fill_between(preproc_metadata['trial_time_axis'], column_scores[igroup]-column_scores_std[igroup],column_scores[igroup], alpha=0.5, color=column_colors[igroup])

[ax[iax].set(xlabel='Time [s]', ylabel='Decoding Accuracy', ylim=(0.125,.5), title=align_event) for iax in range(len(ax))]

column_mean = np.mean(np.vstack(column_scores), axis=0)
column_std = np.mean(np.vstack(column_scores_std), axis=0)
column_max = np.mean(np.vstack(column_scores_max), axis=0)
column_min = np.mean(np.vstack(column_scores_min), axis=0)
ax[1].plot(preproc_metadata['trial_time_axis'], np.mean(column_scores, axis=0), color='k')
ax[1].fill_between(preproc_metadata['trial_time_axis'], column_mean+column_std, column_mean, color='k', alpha=0.5)
ax[1].fill_between(preproc_metadata['trial_time_axis'], column_mean-column_std, column_mean, color='k', alpha=0.5)


for igroup in range(len(column_sites)):
    ax[2].plot(preproc_metadata['trial_time_axis'], np.vstack(column_scores)[igroup,:], color=column_colors[igroup])
    ax[2].fill_between(preproc_metadata['trial_time_axis'], np.vstack(column_scores_max)[igroup,:], np.vstack(column_scores)[igroup,:], color=column_colors[igroup], alpha=0.25)
    ax[2].fill_between(preproc_metadata['trial_time_axis'], np.vstack(column_scores_min)[igroup,:], np.vstack(column_scores)[igroup,:], color=column_colors[igroup], alpha=0.25)
ax[2].set(title=f"{align_event} - {pseudopopulation_metadata['nrandom_groups']} iteration envelope")

plt.suptitle(f"Column-based Unit Groups {align_event}")
fig.tight_layout()
plt.show()

In [ ]:
# Decoding accuracy as a function of ecog maps
fig, ax = plt.subplots(1,1,figsize=(6,5))
align_event = align_events[-1]
column_scores = [np.mean(lda_results[align_event]['pseudopopulations']['column_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
column_site_idx = [igroup for igroup in range(len(column_units)) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
max_idx = np.argmax(np.vstack(column_scores),axis=1)
max_scores = np.array([100*np.vstack(column_scores)[icol,max_idx[icol]] for icol in range(len(column_scores))])
_,_,_,pval,regfit = aopy.analysis.base.linear_fit_analysis2D(np.array(ecog_dec_acc_rec_site_pp[align_event])/np.max(np.array(ecog_dec_acc_rec_site_pp[align_event])), max_scores)
# for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])):
[ax.plot(np.array(ecog_dec_acc_rec_site_pp[align_event])[column_site_idx[icol]]/np.max(np.array(ecog_dec_acc_rec_site_pp[align_event])), 100*np.vstack(column_scores)[icol,max_idx[icol]], '.', color=column_colors[icol], markersize=30) for icol in range(len(column_scores))]
ax.plot([0, 1], [regfit.intercept_[0],regfit.coef_[0][0]+regfit.intercept_[0]], '--', color=(0.5,0.5,0.5))               
# ax[ievent].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (2e7, 60))
ax.set(xlabel='$\mu$ECoG Weight [a.u.]', ylabel='Decoding Accuracy [%]', ylim=(12.5,60),xlim=(0,1.1))
print(pval)
if pval < 0.05:
    ax.annotate('*',(1.05,(2*10)+27), fontsize=30, color=(0.5,0.5,0.5))
else:
    ax.annotate(f'{np.round(pval,3)}',(.95,35), fontsize=30, color=(0.5,0.5,0.5))
    
fig.tight_layout()
aopy.visualization.savefig(base_save_dir, 'lda_column_ecog.svg')

In [ ]:
# Decoding accuracy as a function of ecog maps
fig, ax = plt.subplots(1,1,figsize=(6,5))
align_event = align_events[-1]
column_scores = np.array([np.mean(lda_results[align_event]['pseudopopulations']['column_scores_event'][igroup]) for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores_event'])) if len(lda_results[align_event]['pseudopopulations']['column_scores_event'][igroup])>0])
column_site_idx = [igroup for igroup in range(len(column_units)) if len(lda_results[align_event]['pseudopopulations']['column_scores_event'][igroup])>0]
_,_,_,pval,regfit = aopy.analysis.base.linear_fit_analysis2D(np.array(ecog_dec_acc_rec_site_pp[align_event])/np.max(np.array(ecog_dec_acc_rec_site_pp[align_event])), max_scores)
[ax.plot(np.array(ecog_dec_acc_rec_site_pp[align_event])[column_site_idx[icol]]/np.max(np.array(ecog_dec_acc_rec_site_pp[align_event])), 100*column_scores[icol], '.', color=column_colors[icol], markersize=30) for icol in range(len(column_scores))]
ax.plot([0, 1], [regfit.intercept_[0],regfit.coef_[0][0]+regfit.intercept_[0]], '--', color=(0.5,0.5,0.5))               
# ax[ievent].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (2e7, 60))
ax.set(xlabel='$\mu$ECoG Weight [a.u.]', ylabel='Decoding Accuracy [%]', ylim=(12.5,60),xlim=(0,1.1))
print(pval)
if pval < 0.05:
    ax.annotate('*',(1.05,(2*10)+27), fontsize=30, color=(0.5,0.5,0.5))
else:
    ax.annotate(f'{np.round(pval,3)}',(.95,35), fontsize=30, color=(0.5,0.5,0.5))
    
fig.tight_layout()
# aopy.visualization.savefig(base_save_dir, 'lda_column_ecog.svg')

In [ ]:
# Decoding accuracy as a function of ecog maps
fig, ax = plt.subplots(1,3,figsize=(13.5,4.5))
# for ievent, align_event in enumerate(align_events):
align_event = align_events[-1]
column_scores = [np.mean(lda_results[align_event]['pseudopopulations']['column_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
column_site_idx = [igroup for igroup in range(len(column_units)) if len(lda_results[align_event]['pseudopopulations']['column_scores'][igroup])>0]
max_idx = np.argmax(np.vstack(column_scores),axis=1)
max_scores = np.array([100*np.vstack(column_scores)[icol,max_idx[icol]] for icol in range(len(column_scores))])
_,_,_,pval,regfit = aopy.analysis.base.linear_fit_analysis2D(np.array(ecog_dec_acc_rec_site_pp[align_event])/np.max(np.array(ecog_dec_acc_rec_site_pp[align_event])), max_scores)
# for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])):
[ax[ievent].plot(np.array(ecog_dec_acc_rec_site_pp[align_event])[column_site_idx[icol]]/np.max(np.array(ecog_dec_acc_rec_site_pp[align_event])), 100*np.vstack(column_scores)[icol,max_idx[icol]], '.', color=column_colors[icol], markersize=20) for icol in range(len(column_scores))]
ax[ievent].plot([0, 1], [regfit.intercept_[0],regfit.coef_[0][0]+regfit.intercept_[0]], '--', color=(0.5,0.5,0.5))               
# ax[ievent].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (2e7, 60))
ax[ievent].set(xlabel='$\mu$ECoG Weight [a.u.]', ylabel='Decoding Accuracy [%]', ylim=(12.5,60),xlim=(0,1.1), title=f"{align_event}")
if pval < 0.05:
    ax[ievent].annotate('*',(1.05,(ievent*10)+27), fontsize=30, color=(0.5,0.5,0.5))
    
fig.tight_layout()
aopy.visualization.savefig(base_save_dir, 'lda_column_ecog.svg')

In [ ]:
# Plot unit weight by depth
print(len(lda_results[align_event]['pseudopopulations']['column_weights']))
print(lda_results[align_event]['pseudopopulations']['column_weights'][0].shape) # ntime, ntarget, nfold, niter
# for align_event in align_events:
align_event = align_events[-1]
for igroup in range(len(lda_results[align_event]['pseudopopulations']['column_scores'])):
    weights = np.mean(lda_results[align_event]['pseudopopulations']['column_weights'][igroup], axis=(2,3))

#### Depth

In [ ]:
depth_color_grad = aopy.visualization.get_color_gradient_RGB(len(lda_results[align_event]['pseudopopulations']['depth_scores']), [1,0,0], [.5, .5, .5])

In [ ]:
depth_idx = np.where(groups_of_nunits_all_recs == ndepthunits+1)[0][0]
for align_event in align_events:
    if align_event == align_events[-1]:
        unit_adding_scores = [np.mean(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
        depth_scores = [np.mean(lda_results[align_event]['pseudopopulations']['depth_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores']))]
        depth_scores_std = [np.std(lda_results[align_event]['pseudopopulations']['depth_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores']))]
        depth_scores_max = [np.max(lda_results[align_event]['pseudopopulations']['depth_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores']))]
        depth_scores_min = [np.min(lda_results[align_event]['pseudopopulations']['depth_scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores']))]
        fig, ax = plt.subplots(1,3,figsize=(13,3.5))
    
        [ax[0].plot(preproc_metadata['trial_time_axis'], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup])  for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores']))]
        ax[0].plot(preproc_metadata['trial_time_axis'], np.vstack(unit_adding_scores)[depth_idx,:], 'k')
        ax[0].legend([column_units[igroup]['rec_site'][0] for igroup in range(len(column_units))], bbox_to_anchor=(1,1))
        ax[0].set(xlabel='Time [s]', ylabel='Decoding Accuracy', ylim=(0.125,1), title=align_event)
        ax[0].legend(pseudopopulation_metadata['depth_ranges'], bbox_to_anchor=(1,1))
        # for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores'])):
        #     ax[0].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], depth_scores[igroup]+depth_scores_std[igroup],depth_scores[igroup], alpha=0.5, color=f"C{igroup}")
        #     ax[0].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], depth_scores[igroup]-depth_scores_std[igroup],depth_scores[igroup], alpha=0.5, color=f"C{igroup}")
    
        
        depth_mean = np.mean(np.vstack(depth_scores), axis=0)
        depth_std = np.mean(np.vstack(depth_scores_std), axis=0)
        depth_max = np.mean(np.vstack(depth_scores_max), axis=0)
        depth_min = np.mean(np.vstack(depth_scores_min), axis=0)
        ax[1].plot(np.nanmax(np.vstack(depth_scores)[:,trelevant1:trelevant2],axis=1), np.mean(pseudopopulation_metadata['depth_ranges'], axis=1), color='firebrick') 
        ax[1].set(xlabel='Decoding Accuracy', ylabel='Depth', xlim=(0.3, 1))
        ax[1].plot([np.nanmax(np.vstack(unit_adding_scores)[depth_idx,trelevant1:trelevant2]), np.nanmax(np.vstack(unit_adding_scores)[depth_idx,trelevant1:trelevant2])], ax[1].get_ylim(), 'k')
        ax[1].legend(['Depth', 'random'])
        ax[1].invert_yaxis()
        # ax[1].plot(preproc_metadata['trial_time_axis'], np.nanmean(depth_scores, axis=0), color='k')
        # ax[1].fill_between(preproc_metadata['trial_time_axis'], depth_mean+depth_std, depth_mean, color='k', alpha=0.5)
        # ax[1].fill_between(preproc_metadata['trial_time_axis'], depth_mean-depth_std, depth_mean, color='k', alpha=0.5)
    
    
        for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores'])):
            ax[2].plot(preproc_metadata['trial_time_axis'], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup])
            ax[2].fill_between(preproc_metadata['trial_time_axis'], np.vstack(depth_scores_max)[igroup,:], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup], alpha=0.25)
            ax[2].fill_between(preproc_metadata['trial_time_axis'], np.vstack(depth_scores_min)[igroup,:], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup], alpha=0.25)
        ax[2].set(title=f"{align_event} - {pseudopopulation_metadata['nrandom_groups']} iteration envelope")
        
        plt.suptitle(f"Column-based Unit Groups {align_event}")
        fig.tight_layout()
        plt.show()

In [ ]:
depth_scores_by_site = []
fig, ax =plt.subplots(1,len(list(depth_group_info_by_site.keys())), figsize=(len(list(depth_group_info_by_site.keys()))*3, 3))
for isite, site in enumerate(list(depth_group_info_by_site.keys())):
    temp_depth_scores = [np.mean(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site]))]
    tmaxidx = np.argmax(np.vstack(temp_depth_scores), axis=1)
    depth_score_max = [np.vstack(temp_depth_scores)[idepth,tmaxidx[idepth]] for idepth in range(len(tmaxidx))]
    
    ax[isite].plot(depth_score_max, np.mean(pseudopopulation_metadata['depth_ranges'], axis=1), '*')
    ax[isite].set(xlabel='Decoding Accuracy', ylabel='Depth', title=f"Site: {site}", xlim=(0, 0.9), ylim=(0, 3840))
    ax[isite].invert_yaxis()
fig.tight_layout()
plt.show()


In [ ]:
depth_scores_by_site = []
fig, ax =plt.subplots(1,1, figsize=(4, 6))
site_max_depth = []
site_min_depth = []
site_depth_range = []
xpos_plt = []
depth_colors = []
sites = np.array([column_units[irec]['rec_site'][0] for irec in range(len(column_units))])
xpos = np.array([column_units[irec]['rec_rcaxis'][0] for irec in range(len(column_units))])[np.argsort(sites)]
for isite, site in enumerate(list(depth_group_info_by_site.keys())):
    temp_depth_scores = [np.mean(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site])-1)]
    tmaxidx = np.argmax(np.vstack(temp_depth_scores), axis=1)
    depth_score_max = [100*np.vstack(temp_depth_scores)[idepth,tmaxidx[idepth]] for idepth in range(len(tmaxidx))]

    if np.sum(np.isnan(depth_score_max)) < 8:
        # if isite == 4:
        #     depth_score_max[6] = 36.6380405
        #     # stop
        ax.plot(depth_score_max, np.mean(pseudopopulation_metadata['depth_ranges'], axis=1)[:-1]/1000, color=column_colors[np.argsort(sites)[isite]])
        ax.set(xlabel='Decoding \n Accuracy [%]', ylabel='Depth [mm]', xlim=(20, 65), ylim=(0, 3.840))
        ax.invert_yaxis()
        start_idx = np.where(~np.isnan(depth_score_max))[0][0]
        end_idx = np.where(~np.isnan(depth_score_max))[0][-1]
        site_depth_range.append(depth_score_max[start_idx] - depth_score_max[end_idx])
        # site_max_depth.append(np.mean(pseudopopulation_metadata['depth_ranges'], axis=1)[np.argmax(depth_score_max)])
        # site_min_depth.append(np.mean(pseudopopulation_metadata['depth_ranges'], axis=1)[np.argmin(depth_score_max)])
        xpos_plt.append(xpos[isite])
        # print(site,np.mean(pseudopopulation_metadata['depth_ranges'], axis=1)[np.argmax(depth_score_max)])
        depth_colors.append(column_colors[np.argsort(sites)[isite]])
        

aopy.visualization.savefig(base_save_dir, 'lda_depth_by_site.svg')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4.5))
[ax.plot(xpos_plt[icol],site_depth_range[icol],'.', markersize=20, color=depth_colors[icol]) for icol in range(len(depth_colors))]
# ax[1].plot(xpos_plt, site_min_depth,'.')
# ax[1].legend(['max','min'])
# ax[1].invert_yaxis()
ax.set(xlabel='X-Position [mm]', ylabel='$\Delta$ decoding [%]', xlim=(-6,6))
ax.set(title='Superficial vs. Deep \n Decoding Change')
fig.tight_layout()
aopy.visualization.savefig(base_save_dir, 'lda_depth_dec_diff.svg')

In [ ]:
# depth_idx = np.where(groups_of_nunits_all_recs == ndepthunits)[0][0]
for site in recording_site:
    for align_event in align_events:
        if align_event == align_events[-1]:
            unit_adding_scores = [np.mean(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
            depth_scores = [np.mean(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site]))]
            depth_scores_std = [np.std(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site]))]
            depth_scores_max = [np.max(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site]))]
            depth_scores_min = [np.min(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site][igroup], axis=1) for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site]))]
            fig, ax = plt.subplots(1,3,figsize=(13,3.5))
        
            [ax[0].plot(preproc_metadata['trial_time_axis'], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup])  for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site]))]
            # ax[0].plot(preproc_metadata['trial_time_axis'], np.vstack(unit_adding_scores)[depth_idx,:], 'k')
            ax[0].legend([column_units[igroup]['rec_site'][0] for igroup in range(len(column_units))], bbox_to_anchor=(1,1))
            ax[0].set(xlabel='Time [s]', ylabel='Decoding Accuracy', ylim=(0.125,.8), title=align_event)
            ax[0].legend(pseudopopulation_metadata['depth_ranges'], bbox_to_anchor=(1,1))
            # for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores'])):
            #     ax[0].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], depth_scores[igroup]+depth_scores_std[igroup],depth_scores[igroup], alpha=0.5, color=f"C{igroup}")
            #     ax[0].fill_between(preproc_metadata['trial_time_axis'][nlda_lags:], depth_scores[igroup]-depth_scores_std[igroup],depth_scores[igroup], alpha=0.5, color=f"C{igroup}")
        
            
            depth_mean = np.mean(np.vstack(depth_scores), axis=0)
            depth_std = np.mean(np.vstack(depth_scores_std), axis=0)
            depth_max = np.mean(np.vstack(depth_scores_max), axis=0)
            depth_min = np.mean(np.vstack(depth_scores_min), axis=0)
            ax[1].plot(np.nanmax(np.vstack(depth_scores)[:,trelevant1:trelevant2],axis=1), np.mean(pseudopopulation_metadata['depth_ranges'], axis=1), color='firebrick') 
            ax[1].set(xlabel='Decoding Accuracy', ylabel='Depth', xlim=(0.3, 0.8))
            # ax[1].plot([np.nanmax(np.vstack(unit_adding_scores)[depth_idx,trelevant1:trelevant2]), np.nanmax(np.vstack(unit_adding_scores)[depth_idx,trelevant1:trelevant2])], ax[1].get_ylim(), 'k')
            ax[1].legend(['Depth', 'random'])
            ax[1].invert_yaxis()
            # ax[1].plot(preproc_metadata['trial_time_axis'], np.nanmean(depth_scores, axis=0), color='k')
            # ax[1].fill_between(preproc_metadata['trial_time_axis'], depth_mean+depth_std, depth_mean, color='k', alpha=0.5)
            # ax[1].fill_between(preproc_metadata['trial_time_axis'], depth_mean-depth_std, depth_mean, color='k', alpha=0.5)
        
        
            for igroup in range(len(lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][isite])):
                ax[2].plot(preproc_metadata['trial_time_axis'], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup])
                ax[2].fill_between(preproc_metadata['trial_time_axis'], np.vstack(depth_scores_max)[igroup,:], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup], alpha=0.25)
                ax[2].fill_between(preproc_metadata['trial_time_axis'], np.vstack(depth_scores_min)[igroup,:], np.vstack(depth_scores)[igroup,:], color=depth_color_grad[igroup], alpha=0.25)
            ax[2].set(title=f"{align_event} - {niterations} iteration envelope")
            
            plt.suptitle(f"Column-based Unit Groups {align_event}")
            fig.tight_layout()
            plt.show()

#### High decoding Units

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    ax[ievent].plot(preproc_metadata['trial_time_axis'], lda_results[align_event]['pseudopopulations']['high_dec_scores'])
    ax[ievent].set(xlabel='Time [s]', ylabel='Decoding Accuracy', title=f"{align_event} - Top {nimportant_units} Units", ylim=(0.125, 1))

fig.tight_layout()
plt.show()

In [ ]:
# Waveforms of high decoding units
fig, ax = plt.subplots(1,nimportant_units, figsize=(nimportant_units*2,2))
for iunit in range(nimportant_units):
    ax[iunit].plot(high_decoding_units[-1]['waveform'][iunit])
    ax[iunit].set(xlabel='Time', ylabel='$\mu$V', ylim=(-200, 250), title=f"Site: {high_decoding_units[-1]['rec_site'][iunit]}")
    
fig.tight_layout()
plt.show()

In [ ]:
# Waveforms of low decoding units
fig, ax = plt.subplots(1,nimportant_units, figsize=(nimportant_units*2,2))
for iunit in range(nimportant_units):
    ax[iunit].plot(low_decoding_units[-1]['waveform'][iunit])
    ax[iunit].set(xlabel='Time', ylabel='$\mu$V', ylim=(-200, 250), title=f"Site: {low_decoding_units[-1]['rec_site'][iunit]}")
    
fig.tight_layout()
plt.show()

In [ ]:
    # # Plot FR vs. time traces of the 10 highest weighted neurons
    # avg_fr_each_trial = np.mean(rasters['neural'][align_event][irec][:,:,stable_unit_idx[irec]], axis=0)
    # smooth_avg_fr_each_trial = aopy.analysis.base.calc_rolling_average(avg_fr_each_trial, window_size=17)
    
    # nunits_plot = 8
    # fig, ax = plt.subplots(1,nunits_plot, figsize=(nunits_plot*1.6,2))
    # for iunit in range(nunits_plot):
    #     ax[iunit].plot(smooth_avg_fr_each_trial[:ntest_trials,unit_rank[iunit]])
    #     ax[iunit].set(xlabel='Trial', ylabel='Firing rate', title=f"Unit: {stable_unit_labels[irec][unit_rank[iunit]]}")

    # plt.suptitle(f"Recording {recording_site[irec]} - High weight units")
    # fig.tight_layout()
    # plt.show()

### Trial adding

In [ ]:
# Plot number of stable units as a function of trials for each align_event and recording
### Move to unit quality notebook
# for align_event in align_events:
#     fig, ax = plt.subplots(1,3,figsize=(10,3))
#     for irec in range(nrecs):
#         nstable_units_temp = [len(stable_unit_idx_nunits_tradd[align_event][itradd][irec]) for itradd in range(ntrial_adding_bins)]
#         ax[0].plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, nstable_units_temp, color=day_colors[irec])
#         if recording_site[irec] in recording_brain_areas['M1']:
#             ax[1].plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, nstable_units_temp, color=day_colors[irec])
#         elif recording_site[irec] in recording_brain_areas['PM']:
#             ax[2].plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, nstable_units_temp, color=day_colors[irec])
#     [iax.set(xlabel='Trials', ylabel='Number of Stable Units') for iax in ax]
#     plt.suptitle(f"{align_event} - Neurons active on {100*min_trial_prop}% of trials")
#     fig.tight_layout()
#     plt.show()

In [ ]:
# for iax, align_event in enumerate(align_events):
#     fig, ax = plt.subplots(1,3, figsize=(10,3))
#     ax[0].plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, lda_results_tradd[align_event]['neural_max_pcc_pval'])
#     ax[1].plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, lda_results_tradd[align_event]['neural_max_pcc_pval_M1'])
#     ax[2].plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, lda_results_tradd[align_event]['neural_max_pcc_pval_PM'])
    
#     ax[0].set(xlabel="Trials", ylabel="pvalue", title='All Recordings')
#     ax[1].set(xlabel="Trials", ylabel="pvalue", title="M1")
#     ax[2].set(xlabel="Trials", ylabel="pvalue", title="PM")
    
#     ax0_twinx = ax[0].twinx()
#     ax0_twinx.plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, np.squeeze([lda_results_tradd[align_event]['neural_max_reg_fit'][itradd].coef_ for itradd in range(ntrial_adding_bins)]), 'k')
#     ax0_twinx.set(ylabel='Slope (black)')
    
#     ax1_twinx = ax[1].twinx()
#     ax1_twinx.plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, np.squeeze([lda_results_tradd[align_event]['neural_max_reg_fit_M1'][itradd].coef_ for itradd in range(ntrial_adding_bins)]), 'k')
#     ax1_twinx.set(ylabel='Slope (black)')
    
#     ax2_twinx = ax[2].twinx()
#     ax2_twinx.plot(np.arange(1,ntrial_adding_bins+1)*ntrial_bin_size, np.squeeze([lda_results_tradd[align_event]['neural_max_reg_fit_PM'][itradd].coef_ for itradd in range(ntrial_adding_bins)]), 'k')
#     ax2_twinx.set(ylabel='Slope (black)')
    
#     plt.suptitle(f"{align_event}")
#     fig.tight_layout()
#     plt.show()

### Unit adding

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
unit_adding_scores = [100*np.mean(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
unit_adding_std = [100*np.std(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
unit_adding_max = [100*np.max(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
unit_adding_min = [100*np.min(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
fig, ax = plt.subplots(1,3,figsize=(19.5,4.5))

color_grad = aopy.visualization.get_color_gradient_RGB(len(groups_of_nunits_all_recs_time), [0,0,0], [0.5, 0.5, 0.5])
[ax[0].plot(preproc_metadata['trial_time_axis'], unit_adding_scores[igroup], color=color_grad[igroup,:]) for igroup in range(len(groups_of_nunits_all_recs_time))]
# ax[0].legend([groups_of_nunits_all_recs[igroup] for igroup in range(len(groups_of_nunits_all_recs))], bbox_to_anchor=(1,1))
ax[0].axvline(x=trelevant1_time, color='r', linestyle='--')
ax[0].axvline(x=trelevant2_time, color='r', linestyle='--')
ax[0].set(xlabel='Time from movement [s]', yticks=[20,40,60,80,100], ylim=(0,105))

score_arg_max = np.argmax(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1) + trelevant1
std_plot = [np.vstack(unit_adding_std)[igroup,score_arg_max[igroup]] for igroup in range(len(groups_of_nunits_all_recs_time))]
#[ax[0].plot(preproc_metadata['trial_time_axis'][score_arg_max[igroup]], np.max(np.vstack(unit_adding_scores)[igroup,trelevant1:trelevant2]),'o', color=color_grad[igroup], markersize=15) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
ax[1].plot(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1))
ax[1].fill_between(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1) + std_plot, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), color='C0', alpha=0.5)
ax[1].fill_between(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1) - std_plot, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), color='C0', alpha=0.5)
ax[1].set(xlabel='Number of Units')
ax[1].axhline(y=np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1)[-1], color='r', linestyle='--')

ax[2].scatter(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), s=100, facecolors='none', c=color_grad)
ax[2].set(xlabel='Number of Units')
# ax[2].axhline(y=np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), color='r', linestyle='--')

[ax[iax].set(ylabel='Decoding Accuracy [%]', ylim=(12.5,105), title=align_event) for iax in range(len(ax))]
plt.suptitle(f"All units {align_event}")
fig.tight_layout()

aopy.visualization.savefig(base_save_dir, f'lda_decoding_unit_adding_all_units{align_event}.svg')

In [ ]:
lda_results[align_event]['unit_adding']['scores'][0].shape

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
unit_adding_scores = [100*np.mean(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
unit_adding_std = [100*np.std(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
unit_adding_max = [100*np.max(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
unit_adding_min = [100*np.min(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
fig, ax = plt.subplots(1,1,figsize=(6,4.5))

color_grad = aopy.visualization.get_color_gradient_RGB(len(groups_of_nunits_all_recs_time), [0,0,0], [0.5, 0.5, 0.5])
[ax.plot(preproc_metadata['trial_time_axis'], unit_adding_scores[igroup], color=color_grad[igroup,:]) for igroup in range(len(groups_of_nunits_all_recs_time))]
# ax[0].legend([groups_of_nunits_all_recs[igroup] for igroup in range(len(groups_of_nunits_all_recs))], bbox_to_anchor=(1,1))
# ax.axvline(x=trelevant1_time, color='r', linestyle='--')
# ax.axvline(x=trelevant2_time, color='r', linestyle='--')
ax.set(xlabel='Time from movement [s]', yticks=[20,40,60,80,100], ylim=(0,105))

# score_arg_max = np.argmax(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1) + trelevant1
# std_plot = [np.vstack(unit_adding_std)[igroup,score_arg_max[igroup]] for igroup in range(len(groups_of_nunits_all_recs_time))]
# #[ax[0].plot(preproc_metadata['trial_time_axis'][score_arg_max[igroup]], np.max(np.vstack(unit_adding_scores)[igroup,trelevant1:trelevant2]),'o', color=color_grad[igroup], markersize=15) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
# ax[1].plot(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1))
# ax[1].fill_between(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1) + std_plot, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), color='C0', alpha=0.5)
# ax[1].fill_between(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1) - std_plot, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), color='C0', alpha=0.5)
# ax[1].set(xlabel='Number of Units')
# ax[1].axhline(y=np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1)[-1], color='r', linestyle='--')

# ax[2].scatter(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), s=100, facecolors='none', c=color_grad)
# ax[2].set(xlabel='Number of Units')
# ax[2].axhline(y=np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), color='r', linestyle='--')

ax.set(ylabel='Decoding Accuracy [%]', ylim=(12.5,105))
# plt.suptitle(f"All units {align_event}")
# fig.tight_layout()

aopy.visualization.savefig(base_save_dir, f'{subject}lda_decoding_unit_adding_all_units{align_event}.svg')

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
unit_adding_scores_event = 100*np.array([np.mean(lda_results[align_event]['unit_adding']['scores_event'][igroup]) for igroup in range(len(lda_results[align_event]['unit_adding']['scores_event']))])
unit_adding_scores = 100*np.array([np.mean(lda_results[align_event]['unit_adding']['scores_event'][igroup]) for igroup in range(len(lda_results[align_event]['unit_adding']['scores_event'])) if np.isin(groups_of_nunits_all_recs, groups_of_nunits_all_recs_time)[igroup]])
unit_adding_std_event = 100*np.array([np.std(lda_results[align_event]['unit_adding']['scores_event'][igroup]) for igroup in range(len(lda_results[align_event]['unit_adding']['scores_event']))])
# unit_adding_max = [np.max(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
# unit_adding_min = [np.min(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
fig, ax = plt.subplots(1,1,figsize=(6,4.5))

color_grad = aopy.visualization.get_color_gradient_RGB(len(groups_of_nunits_all_recs_time), [0,0,0], [0.5, 0.5, 0.5])
ax.plot(groups_of_nunits_all_recs, unit_adding_scores_event,'k')
ax.fill_between(groups_of_nunits_all_recs, unit_adding_scores_event - unit_adding_std_event, unit_adding_scores_event + unit_adding_std_event, color='k', alpha=0.5)
# ax.fill_between(groups_of_nunits_all_recs, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1) - std_plot, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1), color='C0', alpha=0.5)
ax.scatter(groups_of_nunits_all_recs_time, unit_adding_scores, s=200, facecolors='none', c=color_grad)
ax.set(xlabel='Number of units', ylabel='Decoding Accuracy [%]',  ylim=(12.5,105))
aopy.visualization.savefig(base_save_dir, f'{subject}_lda_decoding_unit_adding_all_units{align_event}_event.svg')

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
fig, ax = plt.subplots(1,1,figsize=(4,4.25))
unit_adding_scores = [100*np.mean(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
# For each recording location
for igroup in tqdm(range(len(column_units))):

    unit_adding_scores_rec = [100*np.mean(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_std_rec = [100*np.std(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_max_rec = [100*np.max(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_min_rec = [100*np.min(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]

    tempx = []
    tempy = []
    tempstd = []
    tempmax = []
    tempmin = []
    for iunit_group in range(len(unit_adding_scores_rec)):
        tempx.append(groups_of_nunits[iunit_group])
        max_idx = np.argmax(unit_adding_scores_rec[iunit_group][trelevant1:trelevant2]) + trelevant1
        tempy.append(unit_adding_scores_rec[iunit_group][max_idx])
        tempstd.append(unit_adding_std_rec[iunit_group][max_idx])
        tempmax.append(unit_adding_max_rec[iunit_group][max_idx])
        tempmin.append(unit_adding_min_rec[iunit_group][max_idx])
    if column_units[igroup]['rec_site'][0] == 0:
        ax.plot(tempx, tempy, color='k')
    else:
        ax.plot(tempx, tempy, color=column_colors[igroup])
    
ax.plot(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1)[:len(groups_of_nunits)], color=(0.5,0.5,0.5))
ax.set(xlabel='Number of units', ylabel='Decoding Accuracy [%] ', xlim=(0,30))
aopy.visualization.savefig(base_save_dir, f'lda_column_unit_add_{align_event}.svg')

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
fig, ax = plt.subplots(1,1,figsize=(4,4.25))
unit_adding_scores = [100*np.mean(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
# For each recording location
for igroup in tqdm(range(len(column_units))):

    unit_adding_scores_rec = [100*np.mean(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_std_rec = [100*np.std(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_max_rec = [100*np.max(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_min_rec = [100*np.min(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]

    tempx = []
    tempy = []
    tempstd = []
    tempmax = []
    tempmin = []
    for iunit_group in range(len(unit_adding_scores_rec)):
        tempx.append(groups_of_nunits[iunit_group])
        max_idx = np.argmax(unit_adding_scores_rec[iunit_group][trelevant1:trelevant2]) + trelevant1
        tempy.append(unit_adding_scores_rec[iunit_group][max_idx])
        tempstd.append(unit_adding_std_rec[iunit_group][max_idx])
        tempmax.append(unit_adding_max_rec[iunit_group][max_idx])
        tempmin.append(unit_adding_min_rec[iunit_group][max_idx])
    if column_units[igroup]['rec_site'][0] == 0:
        ax.plot(tempx, tempy, color='k')
    else:
        ax.plot(tempx, tempy, color=column_colors[igroup])
    
ax.plot(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1)[:len(groups_of_nunits)], color=(0.5,0.5,0.5))
ax.set(xlabel='Number of units', ylabel='Decoding Accuracy [%] ', xlim=(0,30))
aopy.visualization.savefig(base_save_dir, f'lda_column_unit_add_{align_event}.svg')

In [ ]:
# for align_event in align_events:
align_event = align_events[-1]
fig, ax = plt.subplots(1,3,figsize=(12,4.5))
unit_adding_scores = [100*np.mean(lda_results[align_event]['unit_adding']['scores'][igroup], axis=1) for igroup in range(len(lda_results[align_event]['unit_adding']['scores']))]
# For each recording location
for igroup in tqdm(range(len(column_units))):

    unit_adding_scores_rec = [100*np.mean(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_std_rec = [100*np.std(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_max_rec = [100*np.max(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]
    unit_adding_min_rec = [100*np.min(lda_results[align_event]['unit_adding']['scores_recording'][igroup][iunit_group], axis=1) for iunit_group in range(len(lda_results[align_event]['unit_adding']['scores_recording'][igroup]))]

    tempx = []
    tempy = []
    tempstd = []
    tempmax = []
    tempmin = []
    for iunit_group in range(len(unit_adding_scores_rec)):
        tempx.append(groups_of_nunits[iunit_group])
        max_idx = np.argmax(unit_adding_scores_rec[iunit_group][trelevant1:trelevant2]) + trelevant1
        tempy.append(unit_adding_scores_rec[iunit_group][max_idx])
        tempstd.append(unit_adding_std_rec[iunit_group][max_idx])
        tempmax.append(unit_adding_max_rec[iunit_group][max_idx])
        tempmin.append(unit_adding_min_rec[iunit_group][max_idx])

    ax[0].plot(tempx, tempy, color=column_colors[igroup])
    ax[1].plot(tempx, tempy, color=column_colors[igroup])
    ax[1].fill_between(tempx, np.array(tempy)+np.array(tempstd), tempy, alpha=0.5, color=column_colors[igroup])
    ax[1].fill_between(tempx, np.array(tempy)-np.array(tempstd), tempy, alpha=0.5, color=column_colors[igroup])
    ax[2].fill_between(tempx, tempmax, tempmin, alpha=0.5)
    
ax[0].plot(groups_of_nunits_all_recs_time, np.max(np.vstack(unit_adding_scores)[:,trelevant1:trelevant2], axis=1)[:len(groups_of_nunits)], color=(0.5,0.5,0.5))
plt.suptitle(f"{align_event}")
ax[0].set(xlabel='Number of units', ylabel='Decoding \n Accuracy [%] ', xlim=(0,30))
ax[1].set(xlabel='Number of units', ylabel='Decoding Accuracy', title='Mean +/- std')
ax[2].set(xlabel='Number of units', ylabel='Decoding Accuracy', title='Max - min envelope')
fig.tight_layout()
aopy.visualization.savefig(base_save_dir, f'lda_column_unit_add_{align_event}.svg')

### Spike band power

#### Decoding with time windows

In [ ]:
# All channels for each recording site
fig, ax = plt.subplots(1,len(align_events), figsize=(len(align_events)*4, 3))
for ievent, align_event in enumerate(align_events):
    for irec in range(nrecs):
        ax[ievent].plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['sbp']['scores'][irec], axis=1), color=day_colors[irec])
        ax[ievent].set(xlabel='Time [s]', ylabel='Decoding Accuracy', title=f"{align_event}", ylim=(0.125, 1))

In [ ]:
# Plot weight distribution (independent of location)
# print(lda_results[align_event]['sbp']['weights'][0].shape) # (ntime, ntarget, nch, nfold)
align_event = align_events[-1]
fig, ax = plt.subplots(2,1,figsize=(14,5))
for irec in range(nrecs):
    ax[0].plot(lda_results[align_event]['sbp']['dec_unit_weight_rank'][irec], color=day_colors[irec])
    ax[0].legend(recording_site, bbox_to_anchor=(1,1), fontsize=10)
    ax[1].plot(lda_results[align_event]['sbp']['dec_unit_weight_rank'][irec]/np.max(lda_results[align_event]['sbp']['dec_unit_weight_rank'][irec]), color=day_colors[irec])
    ax[1].plot([10,10], [0,1], 'r--')
    ax[1].plot([20,20], [0,1], 'r--')
    ax[1].plot([50,50], [0,1], 'r--')
    ax[1].plot([200,200], [0,1], 'r--')
    ax[0].set(ylabel='Decoding Weight')
    ax[1].set(xlabel='Ranked Channel', ylabel='Normalized Decoding Weight')

In [ ]:
# Plot heatmap of weight by channel position
temp_depth = 3840-np.sort(np.concatenate((np.arange(0,3840,20), np.arange(0,3840,20))))
fig, ax = plt.subplots(1,2,figsize=(12,4))
all_weights = []
for irec in range(nrecs):
    max_tidx = np.argmax(np.mean(lda_results[align_event]['sbp']['scores'][irec], axis=1))
    ax[0].plot(temp_depth, np.sum(np.abs(np.mean(lda_results[align_event]['sbp']['weights'][irec], axis=3)[max_tidx,:,:]), axis=0), color=day_colors[irec])
    ax[0].set(xlabel='Depth', ylabel='Weight', ylim=(0, 500))
    all_weights.append(np.sum(np.abs(np.mean(lda_results[align_event]['sbp']['weights'][irec], axis=3)[max_tidx,:,:]), axis=0)/np.max(np.sum(np.abs(np.mean(lda_results[align_event]['sbp']['weights'][irec], axis=3)[max_tidx,:,:]), axis=0)))

im_temp = ax[1].pcolor(np.arange(len(recording_site)), temp_depth, np.vstack(all_weights).T) 
cb = plt.colorbar(im_temp)
ax[1].invert_yaxis()
ax[1].set_xticks(np.arange(len(recording_site)), recording_site)
ax[1].set(xlabel='Recording Site', ylabel='Depth')
for xtick, color in zip(ax[1].get_xticklabels(), day_colors):
    xtick.set_color(color)
fig.tight_layout()
plt.show()

In [ ]:
# Just the top 100 channels by total weight magnitude
fig, ax = plt.subplots(1,3, figsize=(12,3))
max_planning_dec_acc_recs = {}
for ievent, align_event in enumerate(align_events):
    max_planning_dec_acc_recs[align_event] = [np.max(np.mean(lda_results[align_event]['sbp']['scores_top_ch'][irec], axis=1)[trelevant1:trelevant2]) for irec in range(nrecs)]
    [ax[ievent].plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['sbp']['scores_top_ch'][irec], axis=1), color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel='Time [s]', ylabel='Neuropixel SBP Dec Acc', title=f"{align_event}: Max: {np.round(np.max(max_planning_dec_acc_recs[align_event]),3)}",ylim=(0.125, 1))
    ax[ievent].legend(recording_site, bbox_to_anchor=(1,1), fontsize=6)
    ax[ievent].plot([0,0], [0,1], 'r--')
    ax[ievent].plot([0.2,0.2], [0,1], 'r--')
    # ax[ievent].annotate(f"Max: {np.round(np.max(max_planning_dec_acc_recs),3)}", (0.25, 0.4))
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events):
    [ax[ievent].plot(ecog_dec_acc_rec_site[align_event][irec], max_planning_dec_acc_recs[align_event][irec], '.', markersize=14, color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel='ECoG Weight', ylabel='SBP Decoding', title=f"{align_event} - Top {nsbp_ch}ch", ylim=(0.125, 1))
fig.tight_layout()
plt.show()

In [ ]:
# Compute delta decoding acc for each site
fig, ax = plt.subplots(1,len(align_events), figsize=(len(align_events)*4,3))
for ievent, align_event in enumerate(align_events):
    [ax[ievent].plot(ecog_dec_acc_rec_site[align_event][irec],max_planning_dec_acc_recs[align_event][irec] - lda_results[align_event]['neural_space_max'][irec], '.', markersize=14, color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel='ECoG Weight', ylabel='(SBP - Unit) Decoding', title=f"{align_event}")

fig.tight_layout()
plt.show()

In [ ]:
align_event = align_events[-1]
fig, ax = plt.subplots(1,nrecs, figsize=(nrecs*3,4))
for irec in range(nrecs):
    cbdsd = ax[irec].pcolor(np.mean(lda_results[align_event]['sbp']['single_ch_decoding'][irec],axis=2).T)
    cb = plt.colorbar(cbdsd)
    ax[irec].set(title=f"{recording_site[irec]}")

fig.tight_layout()
plt.show()

In [ ]:
print(np.mean(lda_results[align_event]['sbp']['single_ch_decoding'][irec],axis=2).shape)
print(np.mean(lda_results[align_event]['sbp']['single_ch_decoding'][irec],axis=2))


In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10,5))
nch = lda_results[align_event]['sbp']['single_ch_decoding'][0].shape[1]
for irec in range(nrecs):
    maxtidx = np.argmax(np.mean(lda_results[align_event]['sbp']['single_ch_decoding'][irec], axis=2), axis=0)
    ax.plot(np.flip(np.sort([np.mean(lda_results[align_event]['sbp']['single_ch_decoding'][irec],axis=2)[maxtidx[ich],ich] for ich in range(nch)])), color=day_colors[irec])
    ax.set(xlabel='Sorted Channel', ylabel='Decoding Accuracy')
fig.tight_layout()
plt.show()

#### Decoding around an event

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,5))
for ievent, align_event in enumerate(align_events):
    max_scores = np.array([np.mean(lda_results[align_event]['sbp']['scores_event'][irec]) for irec in range(nrecs)])
    [ax[ievent].plot(ecog_dec_acc_rec_site[align_event][irec], max_scores[irec], '.', markersize=26, color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel='ECoG Weight', ylabel='SBP Decoding', title=f"{align_event} - All Ch", ylim=(0.125, 1))
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,5))
for ievent, align_event in enumerate(align_events):
    max_scores = np.array([np.mean(lda_results[align_event]['sbp']['scores_top_ch_event'][irec]) for irec in range(nrecs)])
    [ax[ievent].plot(ecog_dec_acc_rec_site[align_event][irec], max_scores[irec], '.', markersize=26, color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel='ECoG Weight', ylabel='SBP Decoding', title=f"{align_event} - Top {nsbp_ch}ch", ylim=(0.125, 1))
fig.tight_layout()
plt.show()

### LFP Power

In [ ]:
lda_results[align_event]['lfp'][iband]['all_units_score'][irec]

In [ ]:
# # All channels for each recording site
# for iband in range(nbands):
#     fig, ax = plt.subplots(1,len(align_events), figsize=(len(align_events)*4, 3))
#     for ievent, align_event in enumerate(align_events):
#         for irec in range(nrecs):
#             ax[ievent].plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['lfp'][iband]['all_units_score'][irec], axis=1), color=day_colors[irec])
#             ax[ievent].set(xlabel='Time [s]', ylabel='Decoding Accuracy', title=f"{align_event}", ylim=(0.125, 0.9))
#     fig.tight_layout()
#     plt.show()

In [ ]:
align_event = align_events[-1]
max_lfp_dec_acc_recs = {}
for iband in range(nbands):
    fig, ax = plt.subplots(1,1, figsize=(4,3))
    # for ievent, align_event in enumerate(align_events):
    max_lfp_dec_acc_recs[iband] = [np.max(np.mean(lda_results[align_event]['lfp'][iband]['scores'][irec], axis=1)[trelevant1:trelevant2]) for irec in range(nrecs)]
    [ax.plot(preproc_metadata['trial_time_axis'], np.mean(lda_results[align_event]['lfp'][iband]['scores'][irec], axis=1), color=day_colors[irec]) for irec in range(nrecs)]
    ax.set(xlabel='Time [s]', ylabel='Neuropixel LFP Dec Acc', title=f"Band: {iband}: Max: {np.round(np.max(max_lfp_dec_acc_recs[iband]),3)}",ylim=(0.125, 1))
    ax.legend(recording_site, bbox_to_anchor=(1,1), fontsize=6)
    ax.plot([0,0], [0,1], 'r--')
    ax.plot([0.2,0.2], [0,1], 'r--')
    # ax[ievent].annotate(f"Max: {np.round(np.max(max_planning_dec_acc_recs),3)}", (0.25, 0.4))
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,nbands, figsize=(3*nbands,3))
for iband in range(nbands):
    [ax[iband].plot(ecog_dec_acc_rec_site[align_event][irec], max_lfp_dec_acc_recs[iband][irec], '.', markersize=14, color=day_colors[irec]) for irec in range(nrecs)]
    ax[iband].set(xlabel='ECoG Weight', ylabel='LFP Power Decoding', title=f"Band {iband}", ylim=(0.125, 0.6))
fig.tight_layout()
plt.show()

In [ ]:
align_event = align_events[-1]
for iband in range(nbands):
    fig, ax = plt.subplots(1,nrecs, figsize=(nrecs*3,4))
    for irec in range(nrecs):
        cbdsd = ax[irec].pcolor(np.mean(lda_results[align_event]['lfp_single_ch'][iband]['scores'][irec],axis=2).T, vmin=0.125, vmax=0.3)
        cb = plt.colorbar(cbdsd)
        ax[irec].set(title=f"{recording_site[irec]}")
        
    plt.suptitle(f"Band: {iband}")
    fig.tight_layout()
    plt.show()

In [ ]:
align_event = align_events[-1]
for iband in range(nbands):
    fig, ax = plt.subplots(1,1, figsize=(10,3.5))
    nch = lda_results[align_event]['lfp_single_ch'][iband]['scores'][0].shape[1]
    for irec in range(nrecs):
        maxtidx = np.argmax(np.mean(lda_results[align_event]['lfp_single_ch'][iband]['scores'][irec], axis=2), axis=0)
        ax.plot(np.flip(np.sort([np.mean(lda_results[align_event]['lfp_single_ch'][iband]['scores'][irec],axis=2)[maxtidx[ich],ich] for ich in range(nch)])), color=day_colors[irec])
        ax.set(xlabel='Sorted Channel', ylabel='Decoding Accuracy', ylim=(0.125,0.3))
    plt.suptitle(f"Band: {iband}")
    fig.tight_layout()
    plt.show()

In [ ]:
align_event = align_events[-1]
for irec in range(nrecs):
    fig, ax = plt.subplots(1,2, figsize=(8,3))
    for iband in range(nbands):
        nch = lda_results[align_event]['lfp_single_ch'][iband]['scores'][0].shape[1]
        maxtidx = np.argmax(np.mean(lda_results[align_event]['lfp_single_ch'][iband]['scores'][irec], axis=2), axis=0)
        ax[0].plot(np.flip(np.sort([np.mean(lda_results[align_event]['lfp_single_ch'][iband]['scores'][irec],axis=2)[maxtidx[ich],ich] for ich in range(nch)])))
        ax[1].plot(scipy.signal.medfilt([np.mean(lda_results[align_event]['lfp_single_ch'][iband]['scores'][irec],axis=2)[maxtidx[ich],ich] for ich in range(nch)],21), 3820-preproc_metadata['ch_ypos'])
        ax[1].invert_yaxis()
    ax[0].set(xlabel='Sorted Channel', ylabel='Decoding Accuracy', ylim=(0.125, 0.3), title=f"Rec Site: {recording_site[irec]}")
    ax[1].set(ylabel='Depth [um]', xlabel='Decoding Accuracy', xlim=(0.125, 0.3), title=f"Rec Site: {recording_site[irec]}")
    ax[0].legend(preproc_metadata['lfp_bands'])
    fig.tight_layout()
    plt.show()

## Linear Regression

### Trial averaged per target

In [ ]:
ifold = 1
for align_event in align_events:
    fig, ax = plt.subplots(1,nrecs,figsize=(3*nrecs,3))
    for irec in range(nrecs):
        unique_targets = np.unique(lin_reg_results[align_event]['test_label_idx_tavg'][ifold])
        [ax[irec].plot(lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][lin_reg_results[align_event]['test_label_idx_tavg'][irec][ifold]==itarget,1], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][lin_reg_results[align_event]['test_label_idx_tavg'][irec][ifold]==itarget,2], '.', color=colors[ii]) for ii, itarget in enumerate(unique_targets)]
        [ax[irec].plot(np.mean(lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][lin_reg_results[align_event]['test_label_idx_tavg'][irec][ifold]==itarget,1], axis=0), np.mean(lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][lin_reg_results[align_event]['test_label_idx_tavg'][irec][ifold]==itarget,2], axis=0), '.', markersize=20, color=colors[ii]) for ii, itarget in enumerate(unique_targets)]
        [ax[irec].plot(np.mean(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][lin_reg_results[align_event]['test_label_idx_tavg'][irec][ifold]==itarget,0], axis=0), np.mean(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][lin_reg_results[align_event]['test_label_idx_tavg'][irec][ifold]==itarget,1], axis=0), '*', markersize=20, color=colors[ii]) for ii, itarget in enumerate(unique_targets)]
        # [ax[irec].set(xlabel="TRD 1 (x)", ylabel="TRD 2 (y)", title=f"Rec Site {recording_site[irec]}", xlim=(-35,35), ylim=(-35,35)) for irec in range(nrecs)]
        [ax[irec].set(xlabel="TRD 1 (x)", ylabel="TRD 2 (y)", title=f"Rec Site {recording_site[irec]}") for irec in range(nrecs)]
    
    plt.suptitle(f"{align_event}")
    fig.tight_layout()
    plt.show()    

In [ ]:
align_event in align_events[-1]
for irec in range(nrecs):
    fig, ax = plt.subplots(2,1,figsize=(15,3))
    ax[0].plot(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,0], label='x velo')
    ax[0].plot(lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,1], label='predicted')
    ax[0].set(xlabel='Trial Number')
    ax[0].legend()
    ax[1].plot(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,1], label='y velo')
    ax[1].plot(lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,2], label='predicted')
    ax[1].set(xlabel='Trial Number')
    ax[1].legend()
    plt.suptitle(f"Recording Site: {recording_site[irec]}")
    fig.tight_layout()
    plt.show()

In [ ]:
align_event in align_events[-1]
for irec in range(nrecs):
    fig, ax = plt.subplots(2,1,figsize=(15,3))
    ax[0].plot(lin_reg_results[align_event]['test_velo_tavg_hand'][irec][ifold][:,0], label='x velo')
    ax[0].plot(lin_reg_results[align_event]['test_pred_velo_tavg_hand'][irec][ifold][:,1], label='predicted')
    ax[0].set(xlabel='Trial Number')
    ax[0].legend()
    ax[1].plot(lin_reg_results[align_event]['test_velo_tavg_hand'][irec][ifold][:,1], label='y velo')
    ax[1].plot(lin_reg_results[align_event]['test_pred_velo_tavg_hand'][irec][ifold][:,2], label='predicted')
    ax[1].set(xlabel='Trial Number')
    ax[1].legend()
    plt.suptitle(f"Recording Site: {recording_site[irec]}")
    fig.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events):    
    tavg_scores = np.array([np.mean(lin_reg_results[align_event]['scores_tavg'][irec], axis=0) for irec in range(nrecs)])
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], tavg_scores[irec,0], '.', color=day_colors[irec]) for irec in range(nrecs)]
    
    # for irec in range(nrecs):
    #     mse = np.mean([sklearn.metrics.mean_squared_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,0], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,0]) for ifold in range(nfolds)])
    #     ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mse, '.', color=day_colors[irec])
    ax[ievent].set(xlabel="ECoG", ylabel="Trial averaged $R^2$", title=align_event)
plt.suptitle(f"X velocity")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events):    
    tavg_scores = np.array([np.mean(lin_reg_results[align_event]['scores_tavg'][irec], axis=0) for irec in range(nrecs)])
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], tavg_scores[irec,1], '.', color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel="ECoG", ylabel="Trial averaged $R^2$", title=align_event, ylim=(-5,1))
plt.suptitle(f"Y velocity")
fig.tight_layout()
plt.show()
    

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,0], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,1]) for ifold in range(nfolds)])
        ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color=day_colors[irec])
    ax[ievent].set(xlabel="ECoG", ylabel="Mean Abs Error", title=align_event)
plt.suptitle(f"X velocity")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,1], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,2]) for ifold in range(nfolds)])
        ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color=day_colors[irec])
    ax[ievent].set(xlabel="ECoG", ylabel="Mean Abs Error", title=align_event)
plt.suptitle(f"Y velocity")
fig.tight_layout()
plt.show()
    

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,0], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,1]) for ifold in range(nfolds)])
        if recording_site[irec] in recording_brain_areas['M1']:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color='purple')
        else:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color='blue')
    ax[ievent].set(xlabel="ECoG", ylabel="Mean Abs Error", title=align_event)
plt.suptitle(f"X velocity")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,1], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,2]) for ifold in range(nfolds)])
        if recording_site[irec] in recording_brain_areas['M1']:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color='purple')
        else:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color='blue')
    ax[ievent].set(xlabel="ECoG", ylabel="Mean Abs Error", title=align_event)
plt.suptitle(f"Y velocity")
fig.tight_layout()
plt.show()
    

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,0], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,1]) for ifold in range(nfolds)])
        if recording_site[irec] in recording_brain_areas['M1']:
            ax[ievent].plot(np.abs(100*np.array(ecog_dec_acc_rec_site_x[align_event])[irec]), mae, '.', color='purple')
        else:
            ax[ievent].plot(np.abs(100*np.array(ecog_dec_acc_rec_site_x[align_event])[irec]), mae, '.', color='blue')
    ax[ievent].set(xlabel="ECoG X", ylabel="Mean Abs Error", title=align_event)
plt.suptitle(f"X velocity vs. X velo ECoG")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,1], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,2]) for ifold in range(nfolds)])
        if recording_site[irec] in recording_brain_areas['M1']:
            ax[ievent].plot(np.abs(100*np.array(ecog_dec_acc_rec_site_y[align_event])[irec]), mae, '.', color='purple')
        else:
            ax[ievent].plot(np.abs(100*np.array(ecog_dec_acc_rec_site_y[align_event])[irec]), mae, '.', color='blue')
    ax[ievent].set(xlabel="ECoG Y", ylabel="Mean Abs Error", title=align_event)
plt.suptitle(f"Y velocity vs. Y velo ECoG")
fig.tight_layout()
plt.show()
    

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae_x = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,0], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,1]) for ifold in range(nfolds)])
        mae_y = np.mean([sklearn.metrics.mean_absolute_error(lin_reg_results[align_event]['test_velo_tavg'][irec][ifold][:,1], lin_reg_results[align_event]['test_pred_velo_tavg'][irec][ifold][:,2]) for ifold in range(nfolds)])
        if recording_site[irec] in recording_brain_areas['M1']:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.sqrt(mae_x**2+mae_y**2), '.', color='purple')
        else:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.sqrt(mae_x**2+mae_y**2), '.', color='blue')
    ax[ievent].set(xlabel="ECoG", ylabel="Mean Abs Error", title=align_event)
plt.suptitle(f"Combined error in X and Y directions")
fig.tight_layout()
plt.show()


#### Neuron Matched

In [ ]:
# Neuron number matched 
# lin_reg_results[align_event]['scores_tavg_nmatch']
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae = np.mean(lin_reg_results[align_event]['mae_tavg_nmatch'][irec])
        if recording_site[irec] in recording_brain_areas['M1']:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color='purple')
        else:
            ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], mae, '.', color='blue')
    
    # Plot trendline/significance
    ax[ievent].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])], [(lin_reg_results[align_event]['tavg_nmatch_reg_fit_M1'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lin_reg_results[align_event]['tavg_nmatch_reg_fit_M1'].intercept_)[0],(lin_reg_results[align_event]['tavg_nmatch_reg_fit_M1'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lin_reg_results[align_event]['tavg_nmatch_reg_fit_M1'].intercept_)[0]], color='purple')               
    ax[ievent].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])], [(lin_reg_results[align_event]['tavg_nmatch_reg_fit_PM'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lin_reg_results[align_event]['tavg_nmatch_reg_fit_PM'].intercept_)[0],(lin_reg_results[align_event]['tavg_nmatch_reg_fit_PM'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lin_reg_results[align_event]['tavg_nmatch_reg_fit_PM'].intercept_)[0]], color='blue')               
    ax[ievent].annotate(f"M1 p: {np.round(lin_reg_results[align_event]['tavg_nmatch_pcc_pval_M1'],3)}", (4e7, 60), color='purple')
    ax[ievent].annotate(f"PM p: {np.round(lin_reg_results[align_event]['tavg_nmatch_pcc_pval_PM'],3)}", (4e7, 70), color='blue')
    
    ax[ievent].set(xlabel="ECoG", ylabel="Mean Abs Error", title=align_event, ylim=(0, 100))
plt.suptitle(f"Combined error in X and Y directions - Neuron Matched")
fig.tight_layout()
plt.show()


In [ ]:
# Neuron number matched 
# lin_reg_results[align_event]['scores_tavg_nmatch']
fig, ax = plt.subplots(1,3, figsize=(12,3))
for ievent, align_event in enumerate(align_events): 
    for irec in range(nrecs):
        mae_dist = lin_reg_results[align_event]['mae_tavg_nmatch'][irec]
        vplts1 = ax[ievent].violinplot(mae_dist, positions=[(100*np.array(ecog_dec_acc_rec_site[align_event]))[irec]], showmedians=True, showmeans=False, widths = 5e6)
        if recording_site[irec] in recording_brain_areas['M1']:            
            vplts1['bodies'][0].set_color('purple')
            vplts1['cbars'].set_colors('purple')
            vplts1['cmedians'].set_colors('purple')
            vplts1['cmins'].set_colors('purple')
            vplts1['cmaxes'].set_colors('purple')
        else:
            vplts1['bodies'][0].set_color('blue')
            vplts1['cbars'].set_colors('blue')
            vplts1['cmedians'].set_colors('blue')
            vplts1['cmins'].set_colors('blue')
            vplts1['cmaxes'].set_colors('blue')
    
    # Plot trendline/significance
    ax[ievent].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])], [(lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_M1'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_M1'].intercept_)[0],(lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_M1'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx])+lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_M1'].intercept_)[0]], color='purple')               
    ax[ievent].plot([100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx]), 100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])], [(lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_PM'].coef_*100*np.min(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_PM'].intercept_)[0],(lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_PM'].coef_*100*np.max(np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx])+lin_reg_results[align_event]['tavg_nmatchdist_reg_fit_PM'].intercept_)[0]], color='blue')               
    ax[ievent].annotate(f"M1 p: {np.round(lin_reg_results[align_event]['tavg_nmatchdist_pcc_pval_M1'],5)}", (5e7, 100), color='purple')
    ax[ievent].annotate(f"PM p: {np.round(lin_reg_results[align_event]['tavg_nmatchdist_pcc_pval_PM'],5)}", (5e7, 120), color='blue')
    
    ax[ievent].set(xlabel="ECoG", ylabel="Mean Abs Error", title=align_event, ylim=(0, 175))
plt.suptitle(f"Combined error in X and Y directions - Neuron Matched")
fig.tight_layout()
plt.show()


### Concatenated trials

#### All units

In [ ]:
ntimeplot = 10000
ifold = 0
align_event = align_events[-1]
for irec in range(nrecs):
    fig, ax = plt.subplots(2,1,figsize=(15,3))
    ax[0].plot(lin_reg_results[align_event]['test_pos'][irec][ifold][:ntimeplot,0], label='x pos')
    ax[0].plot(lin_reg_results[align_event]['test_pred_pos'][irec][ifold][:ntimeplot,1], label='predicted')
    ax[0].legend()
    ax[1].plot(lin_reg_results[align_event]['test_pos'][irec][ifold][:ntimeplot,1], label='y pos')
    ax[1].plot(lin_reg_results[align_event]['test_pred_pos'][irec][ifold][:ntimeplot,2], label='predicted')
    ax[1].legend()
    plt.suptitle(f"Recording Site: {recording_site[irec]}")
    fig.tight_layout()
    plt.show()

In [ ]:
ntimeplot = 10000
ifold = 0
align_event = align_events[-1]
for irec in range(nrecs):
    fig, ax = plt.subplots(2,1,figsize=(15,3))
    ax[0].plot(lin_reg_results[align_event]['test_velo'][irec][ifold][:ntimeplot,0], label='x velo')
    ax[0].plot(lin_reg_results[align_event]['test_pred_velo'][irec][ifold][:ntimeplot,1], label='predicted')
    ax[0].legend()
    ax[1].plot(lin_reg_results[align_event]['test_velo'][irec][ifold][:ntimeplot,1], label='y velo')
    ax[1].plot(lin_reg_results[align_event]['test_pred_velo'][irec][ifold][:ntimeplot,2], label='predicted')
    ax[1].legend()
    plt.suptitle(f"Recording Site: {recording_site[irec]}")
    fig.tight_layout()
    plt.show()

In [ ]:
# Linear regression weights

for align_event in align_events:
    fig, ax = plt.subplots(1,nrecs, figsize=(nrecs*3.5, 3))
    [ax[irec].pcolor(lin_reg_results[align_event]['subspaces_pos'][irec][:,0,:].T,cmap='bwr', vmin=-0.2, vmax=0.2) for irec in range(nrecs)]
    [ax[irec].set(xlabel="unit", ylabel="fold", title=f"{recording_site[irec]}") for irec in range(nrecs)]
    plt.suptitle(f"{align_event} - X-direction")
    fig.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(1,nrecs, figsize=(nrecs*3.5, 3))
    [ax[irec].pcolor(lin_reg_results[align_event]['subspaces_pos'][irec][:,1,:].T,cmap='bwr', vmin=-0.2, vmax=0.2) for irec in range(nrecs)]
    [ax[irec].set(xlabel="unit", ylabel="fold", title=f"{recording_site[irec]}") for irec in range(nrecs)]
    plt.suptitle(f"{align_event} - Y-direction")
    fig.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,3))
[ax.plot(np.sort(np.abs(np.mean(lin_reg_results[align_event]['subspaces_pos'][irec][:,0,:], axis=1))), color=day_colors[irec]) for irec in range(nrecs)]
ax.set(xlabel='Number of units', ylabel='Regression Weight')
plt.show()

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_x = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[0] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_x)[irec], '.',color=day_colors[irec], markersize=8) for irec in range(nrecs)]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', title=align_event, ylim=(-0.35, 0.9))
plt.suptitle(f"Linear regression to x-position")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_y = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[1] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_y)[irec], '.', color=day_colors[irec], markersize=8) for irec in range(nrecs)]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', ylim=(-0.35, 0.9))
plt.suptitle(f"Linear regression to y-position")
fig.tight_layout()
plt.show()

In [ ]:
# Plot x-pos results as a function of number of units
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_x = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[0] for irec in range(nrecs)]
    [ax[ievent].plot(lin_reg_results[align_event]['subspaces_pos'][irec].shape[0], np.array(avg_scores_x)[irec], '.',color=day_colors[irec], markersize=8) for irec in range(nrecs)]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='Number of Units', ylabel='Neuropixel Acc [r^2]', title=align_event, ylim=(-0.35, 0.9))
plt.suptitle(f"Linear regression to x-position")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_y = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[1] for irec in range(nrecs)]
    [ax[ievent].plot(lin_reg_results[align_event]['subspaces_pos'][irec].shape[0], np.array(avg_scores_y)[irec], '.', color=day_colors[irec], markersize=8) for irec in range(nrecs)]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='Number of Units', ylabel='Neuropixel Acc [r^2]', ylim=(-0.35, 0.9))
plt.suptitle(f"Linear regression to y-position")
fig.tight_layout()
plt.show()

In [ ]:
# split by brain area
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_x = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[0] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_x)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_x)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', title=align_event, ylim=(-0.35, 0.9))
plt.suptitle(f"Linear regression to x-position - All units")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_y = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[1] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_y)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_y)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', ylim=(-0.35, 0.9))
plt.suptitle(f"Linear regression to y-position - All units")
fig.tight_layout()
plt.show()

In [ ]:
# split by brain area
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_x = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[0] for irec in range(nrecs)]
    [ax[ievent].plot(lin_reg_results[align_event]['subspaces_pos'][irec].shape[0], np.array(avg_scores_x)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(lin_reg_results[align_event]['subspaces_pos'][irec].shape[0], np.array(avg_scores_x)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='Number of units', ylabel='Neuropixel Acc [r^2]', title=align_event, ylim=(-0.35, 0.85))
plt.suptitle(f"Linear regression to x-position - All units")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_y = [np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=0)[1] for irec in range(nrecs)]
    [ax[ievent].plot(lin_reg_results[align_event]['subspaces_pos'][irec].shape[0], np.array(avg_scores_y)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(lin_reg_results[align_event]['subspaces_pos'][irec].shape[0], np.array(avg_scores_y)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='Number of units', ylabel='Neuropixel Acc [r^2]', ylim=(-0.35, 0.85))
plt.suptitle(f"Linear regression to y-position - All units")
fig.tight_layout()
plt.show()

In [ ]:
# Plot x-velo results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_velo'][irec], axis=0)[0] for irec in range(nrecs)]
    ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event]), np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]')
plt.suptitle(f"Linear regression to x-velocity")
fig.tight_layout()
plt.show()
    
# Plot y-velo results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_velo'][irec], axis=0)[1] for irec in range(nrecs)]
    ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event]), np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]')
plt.suptitle(f"Linear regression to y-velocity")
fig.tight_layout()
plt.show()

#### Neuron matched

In [ ]:
for align_event in align_events:
    fig, ax = plt.subplots(2,nrecs,figsize=(20,6))
    [ax[0,irec].pcolor(lin_reg_results[align_event]['scores_pos_nmatch'][irec][:,0,:].T, vmin=-0.7, vmax=0.7, cmap='bwr') for irec in range(nrecs)]
    [ax[1,irec].pcolor(lin_reg_results[align_event]['scores_pos_nmatch'][irec][:,1,:].T, vmin=-0.7, vmax=0.7, cmap='bwr') for irec in range(nrecs)]
    [ax[0,irec].set(xlabel='Fold', ylabel='Unit group') for irec in range(nrecs)]
    [ax[1,irec].set(xlabel='Fold', ylabel='Unit group') for irec in range(nrecs)]
    [ax[0,irec].set_title(f"Rec site {recording_site[irec]}", color=day_colors[irec]) for irec in range(nrecs)]
    [ax[1,irec].set_title(f"Rec site {recording_site[irec]}", color=day_colors[irec]) for irec in range(nrecs)]
    plt.suptitle(f"{align_event} - $R^2$")
    fig.tight_layout()
    plt.show()

In [ ]:
# Plot score variability across unit group
fig, ax = plt.subplots(1,len(align_events), figsize=(4*len(align_events), 3))
for ievent, align_event in enumerate(align_events):
    [ax[ievent].plot(lin_reg_results[align_event]['subspaces_pos'][irec].shape[0], np.var(np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec][:,0,:], axis=0)),'.', color=day_colors[irec]) for irec in range(nrecs)]
    ax[ievent].set(xlabel='Total number of units', ylabel='$R^2$ variability across unit groups', title=align_event)
fig.tight_layout()
plt.show()

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.median(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[0] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores)[irec], '.', color=day_colors[irec], markersize=8) for irec in range(nrecs)]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', title=align_event)
plt.suptitle(f"Linear regression to x-position")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[1] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores)[irec], '.', color=day_colors[irec], markersize=8) for irec in range(nrecs)]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]')
plt.suptitle(f"Linear regression to y-position")
fig.tight_layout()
plt.show()

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_x = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[0] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_x)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_x)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', title=align_event)
plt.suptitle(f"Linear regression to x-position")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_y = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[1] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_y)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_y)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]')
plt.suptitle(f"Linear regression to y-position")
fig.tight_layout()
plt.show()

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_x = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[0] for irec in range(nrecs)]
    avg_scores_y = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[1] for irec in range(nrecs)]
    avg_scores_both = (np.array(avg_scores_x) + np.array(avg_scores_x))/2
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], avg_scores_both[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], avg_scores_both[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', title=align_event)
plt.suptitle(f"Linear regression to average x and y")
fig.tight_layout()
plt.show()
    

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_x = [np.mean(lin_reg_results[align_event]['scores_velo_nmatch'][irec], axis=(0,2))[0] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_x)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_x)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', title=align_event)
plt.suptitle(f"Linear regression to x-velocity")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores_y = [np.mean(lin_reg_results[align_event]['scores_velo_nmatch'][irec], axis=(0,2))[1] for irec in range(nrecs)]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_y)[irec], '.',color='purple', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[irec], np.array(avg_scores_y)[irec], '.',color='blue', markersize=8) for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]')
plt.suptitle(f"Linear regression to y-velocity")
fig.tight_layout()
plt.show()

#### Split by brain area

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[0] for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx], np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]', title=align_event)
plt.suptitle(f"Linear regression to x-position - M1")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[1] for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[M1_rec_idx], np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [r^2]')
plt.suptitle(f"Linear regression to y-position - M1 ")
fig.tight_layout()
plt.show()

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[0] for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx], np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [$r^2$]', title=align_event)
plt.suptitle(f"Linear regression to x-position - PM")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[1] for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    ax[ievent].plot(100*np.array(ecog_dec_acc_rec_site[align_event])[PM_rec_idx], np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='ECoG Acc [%]', ylabel='Neuropixel Acc [$r^2$]')
plt.suptitle(f"Linear regression to y-position - PM")
fig.tight_layout()
plt.show()

#### KS Drift

In [ ]:
# Plot x-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[0] for irec in range(nrecs)]
    ax[ievent].plot(np.array(ksdrift['drift_max'])[:-1], np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='KS Drift [um]', ylabel='Neuropixel Acc [r^2]', title=align_event)
plt.suptitle(f"Linear regression to x-position")
fig.tight_layout()
plt.show()
    
# Plot y-pos results
fig, ax = plt.subplots(1,3,figsize=(10,3))
for ievent, align_event in enumerate(align_events):
    avg_scores = [np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2))[1] for irec in range(nrecs)]
    ax[ievent].plot(np.array(ksdrift['drift_max'])[:-1], np.array(avg_scores), 'k.', markersize=8)
    # ax[0].plot([100*np.min(ecog_dec_acc_rec_site[align_event]), 100*np.max(ecog_dec_acc_rec_site[align_event])], [(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.min(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0],(lda_results[align_event]['neural_max_reg_fit'].coef_*100*np.max(ecog_dec_acc_rec_site[align_event])+lda_results[align_event]['neural_max_reg_fit'].intercept_)[0]])               
    # ax[0].annotate(f"Regression p: {np.round(lda_results[align_event]['neural_max_pcc_pval'],3)}", (3, 20))
    ax[ievent].set(xlabel='KS Drift [um]', ylabel='Neuropixel Acc [r^2]')
plt.suptitle(f"Linear regression to y-position")
fig.tight_layout()
plt.show()

## LDA vs. Linear regression

In [ ]:
for align_event in align_events:
    fig, ax = plt.subplots(1,3,figsize=(10,3))
    lda_dec_acc = np.array([np.max(np.mean(lda_results[align_event]['scores'][irec],axis=1)) for irec in range(nrecs)])
    lin_reg_r2 = np.array([np.mean(lin_reg_results[align_event]['scores_pos'][irec], axis=(0)) for irec in range(nrecs)])
    ax[0].plot(lda_dec_acc, lin_reg_r2[:,0], '.', label='X')
    ax[0].plot(lda_dec_acc, lin_reg_r2[:,1], '.', label='Y')
    ax[0].set(xlabel='LDA Accuracy', ylabel='Linear Reg $R^2$', title='All')
    ax[0].legend(loc='lower right')
    
    [ax[1].plot(lda_dec_acc[irec], lin_reg_r2[irec,0], '.', label='X', color='royalblue') for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[1].plot(lda_dec_acc[irec], lin_reg_r2[irec,1], '.', label='Y', color='orange') for irec in range(nrecs)  if recording_site[irec] in recording_brain_areas['M1']]
    ax[1].set(xlabel='LDA Accuracy', ylabel='Linear Reg $R^2$', title='M1')
    
    [ax[2].plot(lda_dec_acc[irec], lin_reg_r2[irec,0], '.', label='X', color='royalblue') for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    [ax[2].plot(lda_dec_acc[irec], lin_reg_r2[irec,1], '.', label='Y', color='orange') for irec in range(nrecs)  if recording_site[irec] in recording_brain_areas['PM']]
    ax[2].set(xlabel='LDA Accuracy', ylabel='Linear Reg $R^2$', title='PM')
    
    plt.suptitle(f"{align_event}")
    fig.tight_layout()
    plt.show()
    

In [ ]:
for align_event in align_events:
    fig, ax = plt.subplots(1,3,figsize=(10,3))
    lin_reg_r2 = np.array([np.mean(lin_reg_results[align_event]['scores_pos_nmatch'][irec], axis=(0,2)) for irec in range(nrecs)])
    ax[0].plot(lda_results[align_event]['neural_space_max'], lin_reg_r2[:,0], '.', label='X')
    ax[0].plot(lda_results[align_event]['neural_space_max'], lin_reg_r2[:,1], '.', label='Y')
    ax[0].set(xlabel='LDA Accuracy', ylabel='Linear Reg $R^2$', title='All')
    ax[0].legend(loc='lower right')
    
    [ax[1].plot(lda_results[align_event]['neural_space_max'][irec], lin_reg_r2[irec,0], '.', label='X', color='royalblue') for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['M1']]
    [ax[1].plot(lda_results[align_event]['neural_space_max'][irec], lin_reg_r2[irec,1], '.', label='Y', color='orange') for irec in range(nrecs)  if recording_site[irec] in recording_brain_areas['M1']]
    ax[1].set(xlabel='LDA Accuracy', ylabel='Linear Reg $R^2$', title='M1')
    
    [ax[2].plot(lda_results[align_event]['neural_space_max'][irec], lin_reg_r2[irec,0], '.', label='X', color='royalblue') for irec in range(nrecs) if recording_site[irec] in recording_brain_areas['PM']]
    [ax[2].plot(lda_results[align_event]['neural_space_max'][irec], lin_reg_r2[irec,1], '.', label='Y', color='orange') for irec in range(nrecs)  if recording_site[irec] in recording_brain_areas['PM']]
    ax[2].set(xlabel='LDA Accuracy', ylabel='Linear Reg $R^2$', title='PM')
    
    plt.suptitle(f"{align_event}")
    fig.tight_layout()

# Save decoding accuracies

In [ ]:
save_dir = "/media/moor-data/results/Ryan/neuropixel_targeting/np_analysis_preproc_data"
aopy.data.base.pkl_write(f"{subject}_np_decoding_accuracy", (lda_results[align_event]['neural_space'], recording_site), save_dir)